### sbp_modelo

In [ ]:

import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None 
import os
import sys
import time
import pyodbc
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import plotly.express as px
import lightgbm as lgb
from sklearn import tree
from sklearn import metrics
from sklearn.tree import DecisionTreeRegressor
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_auc_score, cohen_kappa_score
from sklearn.feature_selection import RFE
from IPython.display import display
from tqdm import tqdm_notebook

class MedirTiempoEjecucion:
    def __enter__(self):
        self.start_time = time.time()
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        elapsed_time = time.time() - self.start_time
        print(f"Tiempo transcurrido: {int(elapsed_time // 3600)} horas, {int((elapsed_time % 3600) // 60)} minutos y {elapsed_time % 60:.3f} segundos")
        
#Resumen de las métricas de rendimiento de un modelo de regresión
def metricas_regresion(y_train_real,y_train_predicho,y_test_real,y_test_predicho,plot_actual_vs_predicted=False):
    
    def rmspe(y_true, y_pred):
        return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

    #Train
    mae_train = metrics.mean_absolute_error(y_train_real, y_train_predicho)
    mape_train = metrics.mean_absolute_percentage_error(y_train_real, y_train_predicho)
    mse_train = metrics.mean_squared_error(y_train_real, y_train_predicho)
    rmse_train = metrics.root_mean_squared_error(y_train_real, y_train_predicho)
    rmspe_train = rmspe(y_train_real, y_train_predicho)
    r2_train = metrics.r2_score(y_train_real, y_train_predicho)

    #Test
    mae_test = metrics.mean_absolute_error(y_test_real, y_test_predicho)
    mape_test = metrics.mean_absolute_percentage_error(y_test_real, y_test_predicho)
    mse_test = metrics.mean_squared_error(y_test_real, y_test_predicho)
    rmse_test = metrics.root_mean_squared_error(y_test_real, y_test_predicho)
    rmspe_test = rmspe(y_test_real, y_test_predicho)
    r2_test = metrics.r2_score(y_test_real, y_test_predicho)
    #Resumen
    resultados = { 'MAE': [mae_train, mae_test],'MAPE': [mape_train, mape_test], 'MSE': [mse_train, mse_test], 'RMSE': [rmse_train, rmse_test], 'RMSPE': [rmspe_train, rmspe_test], 'R2': [r2_train, r2_test] }
    resultados = pd.DataFrame(resultados, index=['Train', 'Test'])
    if plot_actual_vs_predicted:
        fig, axs = plt.subplots(1, 2, figsize=(6,3))
        # Primer gráfico:
        display_train = metrics.PredictionErrorDisplay(y_true=y_train_real, y_pred=y_train_predicho)
        display_train.plot(ax=axs[0], kind="actual_vs_predicted")   # actual_vs_predicted / residual_vs_predicted
        axs[0].set_title('Train')
        # Segundo gráfico:
        display_test = metrics.PredictionErrorDisplay(y_true=y_test_real, y_pred=y_test_predicho)
        display_test.plot(ax=axs[1], kind="actual_vs_predicted")    # actual_vs_predicted / residual_vs_predicted
        axs[1].set_title('Test')
        plt.tight_layout()
    return resultados

def metricas_regresion_v2(y_train_real,y_train_predicho,y_test_real,y_test_predicho,y_real_oot1, y_pred_oot1,y_real_oot2, y_pred_oot2,y_real_oot3, y_pred_oot3):

    def rmspe_metric(y_true, y_pred):
        return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

    #   def r2_adj(y_true, y_pred, p):
    #       n = len(y_true)  # asumimos n > p + 1
    #       R2 = metrics.r2_score(y_true, y_pred)
    #       return 1 - (1 - R2) * (n - 1) / (n - p - 1)

    # Mapeo de datasets en el orden requerido
    datasets = [
        ('Train', y_train_real, y_train_predicho),
        ('Test', y_test_real, y_test_predicho),
        ('OOT1', y_real_oot1, y_pred_oot1),
        ('OOT2', y_real_oot2, y_pred_oot2),
        ('OOT3', y_real_oot3, y_pred_oot3),
    ]

    # Calcula métricas en un bucle
    filas = []
    indices = []
    for nombre, y_true, y_pred in datasets:
        mae = metrics.mean_absolute_error(y_true, y_pred)
        mape = metrics.mean_absolute_percentage_error(y_true, y_pred)
        mse = metrics.mean_squared_error(y_true, y_pred)
        rmse = metrics.root_mean_squared_error(y_true, y_pred)
        rmspe = rmspe_metric(y_true, y_pred)
        r2 = metrics.r2_score(y_true, y_pred)
        #   r2aj = r2_adj(y_true, y_pred, p)

        filas.append({
            'MAE': mae,
            'MAPE': mape,
            'MSE': mse,
            'RMSE': rmse,
            'RMSPE': rmspe,
            'R2': r2,
            #   'R2_adj': r2aj
        })
        indices.append(nombre)

    resultados = pd.DataFrame(filas, index=indices)

    return resultados

# Crear matriz de correlación de Spearman de las variables explicativas con el target
def matriz_corr_target(df_variables, target, label_var):
    label_var_1 = list(label_var)
    label_var_1.append(target)
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(2)
    return resultado

# Convertir los valores especiales a valores perdidos
def convertir_valores_especiales_a_missing(df):
    try:
        df = df.replace({-9000001: np.NaN, -9999999: np.NaN, -9000000: np.NaN, -99000632: np.NaN, -99000720: np.NaN, -99000792: np.NaN, -9999998: np.NaN, -9999990: np.NaN,
                        -9999997: np.NaN, -9999996: np.NaN, -9999995: np.NaN, -9999994: np.NaN, -9999993: np.NaN, -9999992: np.NaN, -9999991: np.NaN, -99000800: np.NaN,
                        pd.NA: np.NaN})
    except:
        df = df.replace({-9000001: np.nan, -9999999: np.nan, -9000000: np.nan, -99000632: np.nan, -99000720: np.nan, -99000792: np.nan, -9999998: np.nan, -9999990: np.nan,
                        -9999997: np.nan, -9999996: np.nan, -9999995: np.nan, -9999994: np.nan, -9999993: np.nan, -9999992: np.nan, -9999991: np.nan, -99000800: np.nan,
                        pd.NA: np.nan})

    return df

#Obtiene la lista con las variables constantes
def obtener_var_constantes(df, var_excl=[], umbral_const=0.9999):
    list_var_const = []
    for col in df.columns:
        if col in var_excl:
            continue
        if (df[col].dropna().nunique() == 1) or (df[col].value_counts().iloc[0]/len(df) >= umbral_const):
            list_var_const.append(col)
    return list_var_const

# Crear matriz de correlación de Spearman de las variables explicativas con el target
def matriz_corr_target(df_variables, target, label_var):
    # Filtrar las columnas que existen en el DataFrame
    label_var_1 = [col for col in label_var if col in df_variables.columns]
    label_var_1.append(target)
    
    # Calcular las correlaciones de Spearman
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    
    # Ordenar las columnas por correlación
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    
    # Crear la matriz de correlación
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(2)
    
    return resultado
def matriz_corr_target_v2(df_variables, target, label_var):
    # Filtrar las columnas que existen en el DataFrame
    label_var_1 = [col for col in label_var if col in df_variables.columns]
    label_var_1.append(target)
    
    # Calcular las correlaciones de Spearman
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    
    # Ordenar las columnas por correlación
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    
    # Crear la matriz de correlación
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(2)
    
    # Crear la máscara para la diagonal
    mask = np.zeros_like(resultado, dtype=bool)
    np.fill_diagonal(mask, True)
    
    # Graficar el heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(resultado, annot=True, cmap='Blues', mask=~mask, cbar=False, linewidths=.5, linecolor='black')
    sns.heatmap(resultado, annot=True, cmap='Blues', mask=mask, cbar=True, linewidths=.5, linecolor='black')
    
    return plt.show()

def porcentaje_categorias_plot(data):
    
    category_percentages = {}
    for col in data.columns:
        category_percentages[col] = (data[col].value_counts(normalize=True) * 100).reset_index()
        category_percentages[col].columns = ['Category', 'Percentage']
        category_percentages[col]['Variable'] = col

    category_percentages_df = pd.concat(category_percentages.values())
    
    variables_above_5 = category_percentages_df[category_percentages_df['Percentage'] > 5]['Variable'].unique()
    variables_below_5 = category_percentages_df[category_percentages_df['Percentage'] <= 5]['Variable'].unique()

    category_percentages_df['Order'] = category_percentages_df['Variable'].apply(lambda x: 1 if x in variables_above_5 else 2)

    fig = px.bar(category_percentages_df, y='Variable', x='Percentage', color='Category',
                title='Porcentaje de Categorías por Variable',
                labels={'Percentage': '% de Categoría'},
                category_orders={'Variable': list(variables_above_5) + list(variables_below_5)},
                color_discrete_sequence=px.colors.qualitative.Pastel)

    fig.add_shape(type='line',
                x0=0, y0=len(variables_above_5) - 0.5,
                x1=100, y1=len(variables_above_5) - 0.5,
                line=dict(color='Black', width=3))

    fig.update_layout(barmode='stack', showlegend=False)

    fig.show()

def porcentaje_categorias(data):
    # Lista de variables cualitativas
    variables_with_non_representative_categories = []

    # Bucle para analizar cada variable cualitativa
    for var in data.columns:
        if var in data.columns:  
            freq = data[var].value_counts(normalize=True)  
            dominant_category_percentage = freq.max() 
            non_representative_categories = freq[freq < 0.05]  
    
            if not non_representative_categories.empty:
                num_categories = freq.count()
                num_non_representative_categories = non_representative_categories.count()
                variables_with_non_representative_categories.append((var, num_categories, num_non_representative_categories, 
                                                                    round(dominant_category_percentage*100,2), 
                                                                    round(num_non_representative_categories/num_categories*100,2)))
    
    non_representative = pd.DataFrame(variables_with_non_representative_categories, columns=['Variable', 'Total categorias',
                                                                                            'Total cat. no representativas','% Clase predominante',
                                                                                            '% Cat. no representativas'])
    return non_representative.sort_values(by=['% Clase predominante', '% Cat. no representativas'], ascending=False)

# Muestra las importancias gain y split de un modelo de regresión LightGBM ya entrenado
def importancias_lgbm(modelo,datos):
    split_importance = modelo.feature_importances_
    gain_importance = modelo.booster_.feature_importance(importance_type='gain')

    f_imp = pd.DataFrame({
        'Var': datos.columns,
        'Split_Importance': split_importance,
        'Gain_Importance': gain_importance
    })

    # Calcular el peso de cada importancia
    f_imp['Split_Weight'] = f_imp['Split_Importance'] / f_imp['Split_Importance'].sum() 
    f_imp['Gain_Weight'] = f_imp['Gain_Importance'] / f_imp['Gain_Importance'].sum()
    f_imp['Avg_Weight'] = ( f_imp['Gain_Weight'] + f_imp['Split_Weight'] )/2

    # Ordenar el DataFrame por la importancia de 'gain'
    return f_imp.sort_values("Avg_Weight", ascending=False,ignore_index=True)

def importancias_lgbm_v2(modelo,datos):
    split_importance = modelo.feature_importances_
    gain_importance = modelo.booster_.feature_importance(importance_type='gain')

    f_imp = pd.DataFrame({
        'Var': datos.columns,
        'Split_Importance': split_importance,
        'Gain_Importance': gain_importance
    })

    # Calcular el peso de cada importancia
    f_imp['Split_Weight'] = f_imp['Split_Importance'] / f_imp['Split_Importance'].sum() 
    f_imp['Gain_Weight'] = f_imp['Gain_Importance'] / f_imp['Gain_Importance'].sum()
    f_imp['Avg_Weight'] = ( f_imp['Gain_Weight'] + f_imp['Split_Weight'] )/2
    f_imp['H_Weight'] = 2*f_imp['Gain_Weight']*f_imp['Split_Weight']/( f_imp['Gain_Weight'] + f_imp['Split_Weight'] )
    f_imp['H_Avg_Weight'] = f_imp['H_Weight'] / f_imp['H_Weight'].sum()
    f_imp = f_imp.drop(columns=['H_Weight'])

    # Ordenar el DataFrame por la importancia de 'gain'
    return f_imp.sort_values("H_Avg_Weight", ascending=False,ignore_index=True)

def plot_null_percentage(df):
    """
    Esta función genera una gráfica de barras horizontales que muestra el porcentaje de valores nulos
    por cada variable en un DataFrame dado.
    
    Parámetros:
    df (pandas.DataFrame): El DataFrame para el cual se calculará y visualizará el porcentaje de valores nulos.
    """
    # Calcular el porcentaje de valores nulos por columna
    porcentaje_nulos = df.isnull().mean() * 100
    porcentaje_nulos = porcentaje_nulos.sort_values(ascending=True)

    # Visualizar el porcentaje de valores nulos
    #plt.figure(figsize=(8,6))
    bars = plt.barh(porcentaje_nulos.index, porcentaje_nulos)
    plt.xlabel('Porcentaje')
    plt.title('Porcentaje de valores nulos por cada variable')
    plt.xlim(0, 100)
    
    # Añadir etiquetas con el valor del porcentaje en cada barra
    for bar in bars:
        xval = bar.get_width()
        plt.text(xval + 1, bar.get_y() + bar.get_height()/2, round(xval, 2), va='center', ha='left')
    
    plt.show()
    
def correlacion(df,metodo = "spearman",target=None):
    
    print(".Correlacion "+metodo)
    cor = df.corr(method=metodo)
    df_cor = pd.DataFrame(cor)
    df_cor_abs = np.abs(df_cor)
    
    #Target
    if target is not None:
        df_cor_target = pd.DataFrame(df_cor_abs[target])
        df_cor_target = df_cor_target.sort_values(by = target,ascending = False)
        df_cor_target = df_cor_target.reset_index()
        df_cor_target = df_cor_target.rename(columns={"index":'Variable'})
        df_cor_target = df_cor_target[df_cor_target['Variable'] != target]
        df_cor_target["Top"] = df_cor_target.index
        
        return df_cor, df_cor_abs, df_cor_target
    else:
        return df_cor, df_cor_abs    
def extrae_cortes_arbol(df,cols,arbol):
    if df[cols].dtype == "int64":
        cuts = df[cols].drop_duplicates().to_list()
    else:
        cuts = arbol.tree_.threshold
    cuts = np.unique(cuts)
    #print(cuts)
    #cuts = np.insert(cuts,0,0)
    #cuts = np.insert(cuts,0,0)
    cuts = np.insert(cuts,0,-9999999)
    cuts = np.insert(cuts,0,-9999998)
    cuts = np.insert(cuts,0,-9999990)
    cuts = np.insert(cuts,0,-9000000)
    cuts = np.insert(cuts,0,np.max(cuts))
    cuts = np.sort(cuts)
    cuts = np.delete(cuts, np.where(cuts == -2))
    return cuts
def bin_variable_regresion(df,col,cuts,target,estimado=None):
    n = len(cuts)
    variable = []
    orden_cota = []
    cotas = []
    cota_inf = []
    cota_sup = []
    sql = []
    sql_num = []
    media = []
    if estimado is not None: media_est = []
    conteo = []

    df_count = df[col]
    aux = 1
    #print(n)
    for c in range(n):
        #print(c)
        if aux < 100: orden = ("00"+str(aux))[-2:]
        else: orden = str(aux)
        
        if c == 0: #Primer valor
            df_count = df[df[col] <= cuts[c]]
            cotas_c = orden+". (-, "+str(round(cuts[c],3))+"]"
            cota_inf_c = float('-inf')
            cota_sup_c = cuts[c]
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
            sql_n = "when "+col+" <= "+str(cuts[c])+" then "+str(aux)
        elif c == n-1: #Ultimo valor
            df_count = df[df[col] > cuts[c]] 
            cotas_c = orden+". ("+str(round(cuts[c],3))+", +)"
            cota_inf_c = cuts[c]
            cota_sup_c = float('inf')
            sql_c = "when "+col+" > "+str(cuts[c])+" then '"+cotas_c+"' end ["+col+"_cat]"
            sql_n = "when "+col+" > "+str(cuts[c])+" then "+str(aux)+" end ["+col+"_cat]"
        else: #Valores intermedios
            if (cuts[c-1] < cuts[c]) & (cuts[c] == 0):
                df_count = df[(df[col] < cuts[c])&(df[col] > cuts[c-1])] 
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+")"
            elif (cuts[c-1] == cuts[c]) & (cuts[c] == 0):
                df_count = df[df[col] == cuts[c]]
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            else:
                df_count = df[(df[col] <= cuts[c])&(df[col] > cuts[c-1])] 
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
            sql_n = "when "+col+" <= "+str(cuts[c])+" then "+str(aux)
            cota_inf_c = cuts[c-1]
            cota_sup_c = cuts[c]
        
        if "-9000000.0]" in cotas_c: 
            cotas_c = "Missing"
            sql_c = "when "+col+" is null then '"+cotas_c+"'"
            sql_n = "when "+col+" is null then "+str(aux)

        if aux == 1:
            sql_c = ",case " + sql_c
            sql_n = ",case " + sql_n
            
        media_c = df_count[target].mean()
        if estimado is not None: media_est_c = df_count[estimado].mean()
        conteo_c = len(df_count.index)
        
        if conteo_c > 0:
            variable.append(col)
            orden_cota.append(aux)
            cotas.append(cotas_c)
            cota_inf.append(cota_inf_c)
            cota_sup.append(cota_sup_c)
            sql.append(sql_c)
            sql_num.append(sql_n)
            media.append(media_c)
            if estimado is not None: media_est.append(media_est_c)
            conteo.append(conteo_c)
            aux = aux + 1
        
        if conteo_c == 0 and c == n-1:
            sql[-1] += " end ["+col+"_cat]"
    
    media = [round(x,3) for x in media]
    if estimado is not None: media_est = [round(x,3) for x in media_est]
    cota_inf = [round(x,3) for x in cota_inf]
    cota_sup = [round(x,3) for x in cota_sup]
    conteo_pct = [round(x/sum(conteo),3) for x in conteo]
    
    dict_resumen = ""
    dict_resumen = dict_resumen + "'Variable':variable,'Orden_Cota':orden_cota,'Cotas':cotas,'Cota_Inf':cota_inf,'Cota_Sup':cota_sup,'Media':media"
    if estimado is not None: dict_resumen = dict_resumen + ",'Media_Est':media_est"
    dict_resumen = dict_resumen + ",'Total':conteo,'Total_Pct':conteo_pct,'Sql':sql,'Sql_num':sql_num"
    dict_resumen = "{" + dict_resumen + "}"
    dict_resumen = eval(dict_resumen)
    
    df_resumen = pd.DataFrame(dict_resumen)
    
    return df_resumen
def bin_variable(cuts,df_subset,target,col,orden):
    n = len(cuts)
    variable = []
    orden_var = []
    orden_cota = []
    cotas = []
    sql = []
    media = []
    conteo = []

    df_count = df_subset[col]
    aux = 1
    #print(n)
    for c in range(n):
        #print(c)
        if c == 0: #Primer valor
            df_count = df_subset[df_subset[col] <= cuts[c]]
            cotas_c = str(aux)+". (-, "+str(round(cuts[c],3))+"]"
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
        elif c == n-1: #Ultimo valor
            df_count = df_subset[df_subset[col] > cuts[c]] 
            cotas_c = str(aux)+". ("+str(round(cuts[c],3))+", +)"
            sql_c = "when "+col+" > "+str(cuts[c])+" then '"+cotas_c+"' end ["+col+"_cat]"
        else: #Valores intermedios
            if (cuts[c-1] < cuts[c]) & (cuts[c] == 0):
                df_count = df_subset[(df_subset[col] < cuts[c])&(df_subset[col] > cuts[c-1])] 
                cotas_c = str(aux)+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+")"
            elif (cuts[c-1] == cuts[c]) & (cuts[c] == 0):
                df_count = df_subset[df_subset[col] == cuts[c]]
                cotas_c = str(aux)+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            else:
                df_count = df_subset[(df_subset[col] <= cuts[c])&(df_subset[col] > cuts[c-1])] 
                cotas_c = str(aux)+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
        
        if "-9000000.0]" in cotas_c: 
            cotas_c = cotas_c[:cotas_c.find(".")+1]+" Missing"
            sql_c = "when "+col+" is null then '"+cotas_c+"'"
        
        if aux == 1: sql_c = ",case " + sql_c
        media_c = df_count[target].mean()
        conteo_c = len(df_count.index)
        
        if conteo_c > 0:
            variable.append(col)
            orden_cota.append(aux)
            orden_var.append(orden)
            cotas.append(cotas_c)
            sql.append(sql_c)
            media.append(media_c)
            conteo.append(conteo_c)
            aux = aux + 1
        
        if conteo_c == 0 and c == n-1:
            sql[-1] += " end ["+col+"_cat]"
    
    df_resumen = pd.DataFrame({"Orden":orden_var,"Variable":variable,"Orden_Cota":orden_cota,"Cotas":cotas,"Sql":sql,"Media":media,"Total":conteo})
    df_resumen["Total_Pct"] = (df_resumen["Total"] / df_resumen["Total"].sum()).round(3)
    
    return df_resumen
def descripcion_variable(df,col,orden,redondeo=None):
    
    if redondeo is None: redondeo = 3
    
    # Medidas
    describe = df[col].describe()
    
    if df[col].dtype in (["int64","float64","int32"]):
        iqr = describe["75%"] - describe["25%"]
        outlo = describe["25%"] - 1.5*iqr
        outhi = describe["75%"] + 1.5*iqr
        dict_descripcion = {"Orden":orden
                                    ,"Variable":col
                                    ,"Type":df[col].dtype
                                    ,"Freq":df.shape[0]
                                    ,"Unico":len(df[col].unique())
                                    ,"Media":describe["mean"]
                                    ,"Desv":describe["std"]
                                    ,"Min":describe["min"]
                                    ,"Q1":describe["25%"]
                                    ,"Mediana":describe["50%"]
                                    ,"Q3":describe["75%"]
                                    ,"Max":describe["max"]
                                    ,"Miss":df[col].isna().sum()
                                    ,"Cero":df[col].eq(0).sum()
                                    ,"Pos":df[col].gt(0).sum()
                                    ,"Neg":df[col].lt(0).sum()
                                    ,"OutLo":df[col].lt(outlo).sum()
                                    ,"OutHi":df[col].gt(outhi).sum()
                                    ,"pMiss":(df[col].isna().sum() / df.shape[0]).round(redondeo)
                                    ,"pCero":(df[col].eq(0).sum() / df.shape[0]).round(redondeo)
                                    ,"pPos":(df[col].gt(0).sum() / df.shape[0]).round(redondeo)
                                    ,"pNeg":(df[col].lt(0).sum() / df.shape[0]).round(redondeo)
                                    ,"pOutLo":(df[col].lt(outlo).sum() / df.shape[0]).round(redondeo)
                                    ,"pOutHi":(df[col].gt(outhi).sum() / df.shape[0]).round(redondeo)
                                    ,"Moda":""
                                    ,"nModa":0
                                    ,"pModa":0.0}
    else:
        dict_descripcion = {"Orden":orden
                                    ,"Variable":col
                                    ,"Type":df[col].dtype
                                    ,"Freq":df.shape[0]
                                    ,"Unico":len(df[col].unique())
                                    ,"Media":0
                                    ,"Desv":0
                                    ,"Min":0
                                    ,"Q1":0
                                    ,"Mediana":0
                                    ,"Q3":0
                                    ,"Max":0
                                    ,"Miss":df[col].isna().sum()
                                    ,"Cero":0
                                    ,"Pos":0
                                    ,"Neg":0
                                    ,"OutLo":0
                                    ,"OutHi":0
                                    ,"pMiss":(df[col].isna().sum() / df.shape[0]).round(redondeo)
                                    ,"pCero":0
                                    ,"pPos":0
                                    ,"pNeg":0
                                    ,"pOutLo":0
                                    ,"pOutHi":0
                                    ,"Moda":describe["top"]
                                    ,"nModa":describe["freq"]
                                    ,"pModa":round((describe["freq"] / df.shape[0]),redondeo)}
    
    df_descripcion = pd.DataFrame([dict_descripcion])
    
    return df_descripcion
def extrae_cortes_int(df,col):
    cuts = []
    if df[col].dtype == "int64":
        cuts = df[col].drop_duplicates().to_list()
        cuts = np.unique(cuts)
        cuts = np.insert(cuts,0,-9999999)
        cuts = np.insert(cuts,0,-9999998)
        cuts = np.insert(cuts,0,-9999990)
        cuts = np.insert(cuts,0,-9000000)
        cuts = np.insert(cuts,0,np.max(cuts))
        cuts = np.sort(cuts)
        cuts = np.delete(cuts, np.where(cuts == -2))
    return cuts

def eda(df,cols,target):
    df_target = df[target]
    n = df.shape[0]
    
    df_total = pd.DataFrame(columns=["Orden","Variable","Type","Freq","Unico","Media","Desv","Min","Q1","Mediana","Q3","Max","Miss","Cero","Pos","Neg","OutLo","OutHi","pMiss","pCero","pPos","pNeg","pOutLo","pOutHi","Moda","nModa","pModa"])
    df_variables = pd.DataFrame(columns=["Orden","Variable","Orden_Cota","Cotas","Sql","Media","Total","Total_Pct"])
    
    for i in range(len(cols)):
        #print("...."+str(i+1)+". "+cols[i])
        
        # Resumen Total
        df_descripcion = descripcion_variable(df,cols[i],i+1)
        df_total = pd.concat([df_total,df_descripcion])
        
        if df[cols[i]].dtype in (["int64","float64"]):
            
            try:
                # Base inicial
                df_X = pd.DataFrame(df[cols[i]])
                df_X.loc[df_X[cols[i]].isna(),cols[i]] = -9000000
                df.loc[df[cols[i]].isna(),cols[i]] = -9000000

                # Arbol
                # model = DecisionTreeRegressor(min_samples_split = 0.20,min_samples_leaf = int(n*0.01))#,max_depth = 3
                model = DecisionTreeRegressor(min_samples_split = 0.20,min_samples_leaf = int(n*0.05))#,max_depth = 3
                model.fit(df_X,df_target)

                # Cortes del Arbol
                cuts = extrae_cortes_arbol(df,cols[i],model)
                #print(cuts)
                # Resumen de Variables
                df_bin_variable = bin_variable(cuts,df,target,cols[i],i+1)
                df_variables = pd.concat([df_variables,df_bin_variable])

            except Exception as e:
                print("...."+str(i+1)+". Error: ",e)

    df_variables = df_variables.reset_index()
    
    return df_total, df_variables

def eda_bivariado(df,cols,target,ruta_eda,name=None):
    df_target = df[target]
    n = df.shape[0]
    
    #df_total = pd.DataFrame(columns=["Orden","Variable","Type","Freq","Unico","Media","Desv","Min","Q1","Mediana","Q3","Max","Miss","Cero","Pos","Neg","OutLo","OutHi","pMiss","pCero","pPos","pNeg","pOutLo","pOutHi","Moda","nModa","pModa"])
    df_variables = pd.DataFrame(columns=["Variable","Orden_Cota","Cotas","Sql","Media","Total","Total_Pct"])
    
    #crea_carpeta(".//Exploracion_Variables")
    if name is None:  
        ruta_pdf = ruta_eda+"//02.Analisis_Bivariado.pdf"
    else:
        ruta_pdf = ruta_eda+"//02.Analisis_Bivariado"+name+".pdf"
        
    #df_descripciones = bucle_descripcion_variables(df,cols)
    
    with PdfPages(ruta_pdf) as pdf:    
        for i in range(len(cols)):
            print("...."+str(i+1)+". "+cols[i])

            # Resumen Total
            #df_total = pd.concat([df_total,df_descripcion])

            if df[cols[i]].dtype in (["int64","float64"]):

                try:
                    # Base inicial
                    df_X = pd.DataFrame(df[cols[i]])
                    df_X.loc[df_X[cols[i]].isna(),cols[i]] = -9000000
                    df.loc[df[cols[i]].isna(),cols[i]] = -9000000

                    # Arbol
                    arbol = DecisionTreeRegressor(min_samples_split = 0.20,min_samples_leaf = int(n*0.05))#,max_depth = 3
                    arbol.fit(df_X,df_target)

                    # Cortes del Arbol
                    if df[cols[i]].dtype == "int64":
                        cuts = extrae_cortes_int(df,cols[i])
                    else:
                        cuts = extrae_cortes_arbol(df,cols[i],arbol)
                    
                    # Resumen de Variables
                    df_bin_variable = bin_variable_regresion(df,cols[i],cuts,target)
                    df_variables = pd.concat([df_variables,df_bin_variable])

                    # Graficos
                    # Grafico de Barras
                    plt.figure(figsize=(15, 6), dpi=100)

                    plt.subplot(121)
                    plt.bar(df_bin_variable["Orden_Cota"],df_bin_variable["Total"],width=.5)
                    plt.xticks(df_bin_variable["Orden_Cota"])
                    plt.title(str(i+1)+". "+cols[i])
                    axes2 = plt.twinx()
                    axes2.plot(df_bin_variable["Orden_Cota"], df_bin_variable['Media'], color='Orange',marker = "o")

                    # Tabla resumen
                    plt.subplot(122)
                    cell_text = []
                    df_X = df_bin_variable[["Cotas", "Media", "Total","Total_Pct"]]
                    df_X.loc[:,"Media"] = df_X["Media"].apply(lambda x: round(x,3))
                    df_X.loc[:,"Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    #df_X["Media"] = df_X["Media"].apply(lambda x: round(x,1))
                    #df_X["Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    for row in range(len(df_X)):
                        cell_text.append(df_X.iloc[row])

                    tab = plt.table(cellText=cell_text, colLabels=df_X.columns, cellLoc='center',loc='center',colWidths=[0.6, 0.2,0.2,0.2])
                    tab.auto_set_font_size(False)
                    #tab.set_fontsize(12)
                    #tab.scale(2, 2)
                    plt.axis('off')
                    plt.tight_layout()
                    plt.subplots_adjust(left=0.1,bottom=0.1,right=0.9,top=0.9,wspace=0.25,hspace=0.4)

                    pdf.savefig()
                    plt.close()

                    # Arbol
                    fig = plt.figure(figsize=(25,20))
                    _ = tree.plot_tree(arbol, 
                                    feature_names=[cols[i]],
                                    #class_names=iris.target_names,
                                    filled=True)
                    pdf.savefig()
                    plt.close()
                except Exception as e:
                    print("...."+str(i+1)+". Error: ",e)

    df_variables = df_variables.reset_index()
    
    #return df_descripciones, df_variables    

def eda_vf(df,target,var_excluidas,ruta_eda):

    start_time = time.time()
    
    df1 = df.drop(columns=var_excluidas) 
    df2 = df1

    # Correlación
    df_cor, df_cor_abs, df_cor_target = correlacion(df1,target=target)

    print("..Eda")
    #Extrae top de correlacion y columnas faltantes
    cols = df1.columns.to_list()
    cols.sort()
    cols.remove(target)
    df_cols = pd.DataFrame(cols)
    df_cols = df_cols.rename({0: "Variable"}, axis=1)
    df_cols = df_cor_target.merge(df_cols,how="outer",on="Variable")
    cols = df_cols["Variable"].to_list()

    #Descripcion Variables
    tops = df_cols["Top"].to_list()

    df_descripcion = pd.DataFrame()
    for i in range(len(cols)):
        df_descripcion = pd.concat([df_descripcion,descripcion_variable(df1,cols[i],tops[i],redondeo=None)])

    #Inicia EDA
    df1, df_variables = eda(df1,cols,target)

    with pd.ExcelWriter(ruta_eda+"\\01.Analisis_Exploratorio.xlsx") as writer:  
        df_cor.to_excel(writer, sheet_name='Correlacion')
        df_cor_abs.to_excel(writer, sheet_name='Correlacion_Abs')
        df_cor_target.to_excel(writer, sheet_name='Correlacion_Target',index=False)
        df_descripcion.to_excel(writer, sheet_name='EDA',index=False)
        df_variables.to_excel(writer, sheet_name='Variables',index=False)
        #df.to_excel(writer,sheet_name='diccionario',index=False)

    #Inicia EDA bivariado
    print("...Bivariado:")
    eda_bivariado(df2,cols,target,ruta_eda)   
    elapsed_time = time.time() - start_time
    print(f"Tiempo transcurrido: {int(elapsed_time // 60)} minutos y {elapsed_time % 60:.3f} segundos")
    start_time = time.time()

def eda_vf2(df,target,var_excluidas,var_categoricas,ruta_eda):

    start_time = time.time()
    
    df = df.drop(columns=var_excluidas) # quitando variables
    df1 = df.drop(columns=var_categoricas) 
    df2 = df1
    df_cat = df[[target]+var_categoricas]

    # Correlación
    df_cor, df_cor_abs, df_cor_target = correlacion(df1,target=target)

    print("..Eda")
    #Extrae top de correlacion y columnas faltantes
    cols = df1.columns.to_list()
    cols.sort()
    cols.remove(target)
    df_cols = pd.DataFrame(cols)
    df_cols = df_cols.rename({0: "Variable"}, axis=1)
    df_cols = df_cor_target.merge(df_cols,how="outer",on="Variable")
    cols = df_cols["Variable"].to_list()

    #Descripcion Variables
    tops = df_cols["Top"].to_list()

    df_descripcion = pd.DataFrame()
    for i in range(len(cols)):
        df_descripcion = pd.concat([df_descripcion,descripcion_variable(df1,cols[i],tops[i],redondeo=None)])

    #Inicia EDA
    df1, df_variables = eda(df1,cols,target)

    ########################################################################
    # Para var. categóricas

    var_cat = df_cat.columns.to_list()
    var_cat.sort()
    var_cat.remove(target)
    df_cols_cat = pd.DataFrame(var_cat)
    df_cols_cat = df_cols_cat.rename({0: "Variable"}, axis=1)
    var_cat = df_cols_cat["Variable"].to_list()

    #Descripcion Variables cat.
    df_descripcion_cat = pd.DataFrame()
    for i in range(len(var_cat)):
        df_descripcion_cat = pd.concat([df_descripcion_cat,descripcion_variable(df_cat,var_cat[i],var_cat[i],redondeo=None)])

    ########################################################################

    with pd.ExcelWriter(ruta_eda+"\\01.Analisis_Exploratorio.xlsx") as writer:  
        df_cor.to_excel(writer, sheet_name='Correlacion')
        df_cor_abs.to_excel(writer, sheet_name='Correlacion_Abs')
        df_cor_target.to_excel(writer, sheet_name='Correlacion_Target',index=False)
        df_descripcion.to_excel(writer, sheet_name='EDA',index=False)
        df_descripcion_cat.to_excel(writer, sheet_name='EDA_cat',index=False)
        df_variables.to_excel(writer, sheet_name='Variables',index=False)
        #df.to_excel(writer,sheet_name='diccionario',index=False)

    #Inicia EDA bivariado
    print("...Bivariado:")
    eda_bivariado(df2,cols,target,ruta_eda)   
    elapsed_time = time.time() - start_time
    print(f"Tiempo transcurrido: {int(elapsed_time // 60)} minutos y {elapsed_time % 60:.3f} segundos")
    start_time = time.time()

def coincidencia(y_real_train, y_pred_train, umbral, index):
    
    porc_pres_train = ((y_pred_train - y_real_train)/y_real_train)

    # Definir las condiciones para la clasificación
    condiciones_train = [
        (porc_pres_train >= -umbral) & (porc_pres_train <= umbral),  # Coincidencia
        porc_pres_train > umbral,  # Sobreestimación
        porc_pres_train < -umbral  # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Crea el vector de clasificación
    clasificacion_train = np.select(condiciones_train, clases, default='No clasificado')

    # Crear DataFrames con los resultados
    df_precision_train = pd.DataFrame({'y_real': y_real_train, 'y_predicho': y_pred_train, 'porc_pres': porc_pres_train, 'clas_pres': clasificacion_train})
    
    # Calcular las proporciones de cada clase
    presc_train = df_precision_train['clas_pres'].value_counts(normalize=True)

    # Crear un DataFrame con los resultados finales
    resultados = {
        'Subestimacion': [presc_train.get('Subestimacion', 0)],
        'Coincidencia': [presc_train.get('Coincidencia', 0)],
        'Sobreestimacion': [presc_train.get('Sobreestimacion', 0)]
    }
    df_resultados_precs = pd.DataFrame(resultados, index=[index])
    
    return df_resultados_precs

def calcular_coincidencia(y_pred_train, y_pred_test, y_real_train, y_real_test, umbral):
    # Evitar errores dividiendo por cero
    #y_real_train = np.where(y_real_train == 0, np.nan, y_real_train)
    #y_real_test = np.where(y_real_test == 0, np.nan, y_real_test)
    
    # Calcular el porcentaje de error
    porc_pres_train = ((y_pred_train - y_real_train)/y_real_train)
    porc_pres_test = ((y_pred_test - y_real_test)/y_real_test)

    # Definir las condiciones para la clasificación
    condiciones_train = [
        (porc_pres_train >= -umbral) & (porc_pres_train <= umbral),  # Coincidencia
        porc_pres_train > umbral,  # Sobreestimación
        porc_pres_train < -umbral  # Subestimación
    ]
    condiciones_test = [
        (porc_pres_test >= -umbral) & (porc_pres_test <= umbral),  # Coincidencia
        porc_pres_test > umbral,  # Sobreestimación
        porc_pres_test < -umbral  # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Crea el vector de clasificación
    clasificacion_train = np.select(condiciones_train, clases, default='No clasificado')
    clasificacion_test = np.select(condiciones_test, clases, default='No clasificado')

    # Crear DataFrames con los resultados
    df_precision_train = pd.DataFrame({'y_real': y_real_train, 'y_predicho': y_pred_train, 'porc_pres': porc_pres_train, 'clas_pres': clasificacion_train})
    df_precision_test = pd.DataFrame({'y_real': y_real_test, 'y_predicho': y_pred_test, 'porc_pres': porc_pres_test, 'clas_pres': clasificacion_test})

    # Calcular las proporciones de cada clase
    presc_train = df_precision_train['clas_pres'].value_counts(normalize=True)
    presc_test = df_precision_test['clas_pres'].value_counts(normalize=True)

    # Crear un DataFrame con los resultados finales
    resultados = {
        'Subestimacion': [presc_train.get('Subestimacion', 0), presc_test.get('Subestimacion', 0)],
        'Coincidencia': [presc_train.get('Coincidencia', 0), presc_test.get('Coincidencia', 0)],
        'Sobreestimacion': [presc_train.get('Sobreestimacion', 0), presc_test.get('Sobreestimacion', 0)]
    }
    df_resultados_precs = pd.DataFrame(resultados, index=['Train', 'Test'])
    
    return df_resultados_precs

def calcular_coincidencia_v2(y_real_train,y_pred_train, y_real_test,y_pred_test, y_real_oot1,y_pred_oot1, y_real_oot2,y_pred_oot2, y_real_oot3,y_pred_oot3, umbral):

    # Calcular el porcentaje de error
    porc_pres_train = ((y_pred_train - y_real_train)/y_real_train)
    porc_pres_test = ((y_pred_test - y_real_test)/y_real_test)
    porc_pres_oot1 = ((y_pred_oot1 - y_real_oot1)/y_real_oot1)
    porc_pres_oot2 = ((y_pred_oot2 - y_real_oot2)/y_real_oot2)
    porc_pres_oot3 = ((y_pred_oot3 - y_real_oot3)/y_real_oot3)

    # Definir las condiciones para la clasificación
    condiciones_train = [
        (porc_pres_train >= -umbral) & (porc_pres_train <= umbral),  # Coincidencia
        porc_pres_train > umbral,  # Sobreestimación
        porc_pres_train < -umbral  # Subestimación
    ]
    condiciones_test = [
        (porc_pres_test >= -umbral) & (porc_pres_test <= umbral),  # Coincidencia
        porc_pres_test > umbral,  # Sobreestimación
        porc_pres_test < -umbral  # Subestimación
    ]
    condiciones_oot1 = [
        (porc_pres_oot1 >= -umbral) & (porc_pres_oot1 <= umbral),  # Coincidencia
        porc_pres_oot1 > umbral,  # Sobreestimación
        porc_pres_oot1 < -umbral  # Subestimación
    ]
    condiciones_oot2 = [
        (porc_pres_oot2 >= -umbral) & (porc_pres_oot2 <= umbral),  # Coincidencia
        porc_pres_oot2 > umbral,  # Sobreestimación
        porc_pres_oot2 < -umbral  # Subestimación
    ]
    condiciones_oot3 = [
        (porc_pres_oot3 >= -umbral) & (porc_pres_oot3 <= umbral),  # Coincidencia
        porc_pres_oot3 > umbral,  # Sobreestimación
        porc_pres_oot3 < -umbral  # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Crea el vector de clasificación
    clasificacion_train = np.select(condiciones_train, clases, default='No clasificado')
    clasificacion_test = np.select(condiciones_test, clases, default='No clasificado')
    clasificacion_oot1 = np.select(condiciones_oot1, clases, default='No clasificado')
    clasificacion_oot2 = np.select(condiciones_oot2, clases, default='No clasificado')
    clasificacion_oot3 = np.select(condiciones_oot3, clases, default='No clasificado')

    # Crear DataFrames con los resultados
    df_precision_train = pd.DataFrame({'y_real': y_real_train, 'y_predicho': y_pred_train, 'porc_pres': porc_pres_train, 'clas_pres': clasificacion_train})
    df_precision_test = pd.DataFrame({'y_real': y_real_test, 'y_predicho': y_pred_test, 'porc_pres': porc_pres_test, 'clas_pres': clasificacion_test})
    df_precision_oot1 = pd.DataFrame({'y_real': y_real_oot1, 'y_predicho': y_pred_oot1, 'porc_pres': porc_pres_oot1, 'clas_pres': clasificacion_oot1})
    df_precision_oot2 = pd.DataFrame({'y_real': y_real_oot2, 'y_predicho': y_pred_oot2, 'porc_pres': porc_pres_oot2, 'clas_pres': clasificacion_oot2})
    df_precision_oot3 = pd.DataFrame({'y_real': y_real_oot3, 'y_predicho': y_pred_oot3, 'porc_pres': porc_pres_oot3, 'clas_pres': clasificacion_oot3})

    # Calcular las proporciones de cada clase
    presc_train = df_precision_train['clas_pres'].value_counts(normalize=True)
    presc_test = df_precision_test['clas_pres'].value_counts(normalize=True)
    presc_oot1= df_precision_oot1['clas_pres'].value_counts(normalize=True)
    presc_oot2 = df_precision_oot2['clas_pres'].value_counts(normalize=True)
    presc_oot3 = df_precision_oot3['clas_pres'].value_counts(normalize=True)

    # Crear un DataFrame con los resultados finales
    resultados = {
        'Subestimacion': [presc_train.get('Subestimacion', 0), presc_test.get('Subestimacion', 0), presc_oot1.get('Subestimacion', 0), presc_oot2.get('Subestimacion', 0), presc_oot3.get('Subestimacion', 0)],
        'Coincidencia': [presc_train.get('Coincidencia', 0), presc_test.get('Coincidencia', 0), presc_oot1.get('Coincidencia', 0), presc_oot2.get('Coincidencia', 0), presc_oot3.get('Coincidencia', 0)],
        'Sobreestimacion': [presc_train.get('Sobreestimacion', 0), presc_test.get('Sobreestimacion', 0), presc_oot1.get('Sobreestimacion', 0), presc_oot2.get('Sobreestimacion', 0), presc_oot3.get('Sobreestimacion', 0)]
    }
    df_resultados_precs = pd.DataFrame(resultados, index=['Train', 'Test', 'OOT1', 'OOT2', 'OOT3'])
    
    return df_resultados_precs

def calcular_coincidencia_v0(y_pred_train, y_real_train, umbral):
    
    # Calcular el porcentaje de error
    porc_pres_train = ((y_pred_train - y_real_train)/y_real_train)

    # Definir las condiciones para la clasificación
    condiciones_train = [
        (porc_pres_train >= -umbral) & (porc_pres_train <= umbral),  # Coincidencia
        porc_pres_train > umbral,  # Sobreestimación
        porc_pres_train < -umbral  # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Crea el vector de clasificación
    clasificacion_train = np.select(condiciones_train, clases, default='No clasificado')

    # Crear DataFrames con los resultados
    df_precision_train = pd.DataFrame({'y_real': y_real_train, 'y_predicho': y_pred_train, 'porc_pres': porc_pres_train, 'clas_pres': clasificacion_train})

    # Calcular las proporciones de cada clase
    presc_train = df_precision_train['clas_pres'].value_counts(normalize=True)

    # Crear un DataFrame con los resultados finales
    resultados = {
        'Subestimacion': [presc_train.get('Subestimacion', 0)],
        'Coincidencia': [presc_train.get('Coincidencia', 0)],
        'Sobreestimacion': [presc_train.get('Sobreestimacion', 0)]
    }
    df_resultados_precs = pd.DataFrame(resultados, index=['Train'])
    
    return df_resultados_precs

def calcular_precision_x_intervalos(y_pred, y_real, umbral, bins):
    import numpy as np
    import pandas as pd

    porc_pres = (y_pred - y_real) / y_real
    condiciones = [
        (porc_pres < -umbral),              # Subestimación
        (porc_pres >= -umbral) & (porc_pres <= umbral),  # Coincidencia
        (porc_pres > umbral)                # Sobreestimación
    ]
    clases = ['Subestimacion', 'Coincidencia', 'Sobreestimacion']

    clasificacion = np.select(condiciones, clases, default='No clasificado')
    df_precision = pd.DataFrame({
        'y_real': y_real,
        'y_predicho': y_pred,
        'porc_pres': porc_pres,
        'clas_pres': clasificacion
    })

    df_precision['Intervalos'] = pd.cut(df_precision['y_real'], bins)

    frecuencia = df_precision['Intervalos'].value_counts().sort_index()
    porcentaje = frecuencia / frecuencia.sum()
    prom_real= df_precision.groupby('Intervalos')['y_real'].mean()
    prom_pred = df_precision.groupby('Intervalos')['y_predicho'].mean()

    resultado = pd.DataFrame({
        'Casos': frecuencia,
        'Casos (%)': porcentaje,
        'Prom. real': prom_real,
        'Prom. pred': prom_pred
    })

    distribucion = df_precision.groupby(['Intervalos', 'clas_pres']).size().unstack(fill_value=0)

    # Asegurar el orden deseado de las columnas
    for col in ['Subestimacion', 'Coincidencia', 'Sobreestimacion']:
        if col not in distribucion.columns:
            distribucion[col] = 0
    distribucion = distribucion[['Subestimacion', 'Coincidencia', 'Sobreestimacion']]

    distribucion = distribucion.div(distribucion.sum(axis=1), axis=0)

    resultado_final = resultado.join(distribucion, on='Intervalos')

    return resultado_final

def calcular_precision_x_intervalos_v2(y_pred, y_real, umbral, bins):

    def rmspe(y_true, y_pred):
        return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

    porc_pres = (y_pred - y_real) / y_real
    condiciones = [
        (porc_pres < -umbral),              # Subestimación
        (porc_pres >= -umbral) & (porc_pres <= umbral),  # Coincidencia
        (porc_pres > umbral)                # Sobreestimación
    ]
    clases = ['Subestimacion', 'Coincidencia', 'Sobreestimacion']

    clasificacion = np.select(condiciones, clases, default='No clasificado')
    df_precision = pd.DataFrame({
        'y_real': y_real,
        'y_predicho': y_pred,
        'porc_pres': porc_pres,
        'clas_pres': clasificacion
    })

    df_precision['Intervalos'] = pd.cut(df_precision['y_real'], bins)

    frecuencia = df_precision['Intervalos'].value_counts().sort_index()
    porcentaje = frecuencia / frecuencia.sum()
    prom_real= df_precision.groupby('Intervalos')['y_real'].mean()
    prom_pred = df_precision.groupby('Intervalos')['y_predicho'].mean()
    rmspe_cat = df_precision.groupby('Intervalos').apply(lambda df: rmspe(df['y_real'], df['y_predicho']) )

    resultado = pd.DataFrame({
        'Casos': frecuencia,
        'Casos (%)': porcentaje,
        'Prom. real': prom_real,
        'Prom. pred': prom_pred,
        'RMSPE': rmspe_cat
    })

    distribucion = df_precision.groupby(['Intervalos', 'clas_pres']).size().unstack(fill_value=0)

    # Asegurar el orden deseado de las columnas
    for col in ['Subestimacion', 'Coincidencia', 'Sobreestimacion']:
        if col not in distribucion.columns:
            distribucion[col] = 0
    distribucion = distribucion[['Subestimacion', 'Coincidencia', 'Sobreestimacion']]

    distribucion = distribucion.div(distribucion.sum(axis=1), axis=0)

    resultado_final = resultado.join(distribucion, on='Intervalos')

    return resultado_final

def calcular_precision_x_intervalos_v3(y_pred, y_real, umbral, bins):

    from sklearn import metrics

    def rmspe(y_true, y_pred):
        return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

    porc_pres = (y_pred - y_real) / y_real
    condiciones = [
        (porc_pres < -umbral),              # Subestimación
        (porc_pres >= -umbral) & (porc_pres <= umbral),  # Coincidencia
        (porc_pres > umbral)                # Sobreestimación
    ]
    clases = ['Subestimacion', 'Coincidencia', 'Sobreestimacion']

    clasificacion = np.select(condiciones, clases, default='No clasificado')
    df_precision = pd.DataFrame({
        'y_real': y_real,
        'y_predicho': y_pred,
        'porc_pres': porc_pres,
        'clas_pres': clasificacion
    })

    df_precision['Intervalos'] = pd.cut(df_precision['y_real'], bins)

    frecuencia = df_precision['Intervalos'].value_counts().sort_index()
    porcentaje = frecuencia / frecuencia.sum()
    prom_real= df_precision.groupby('Intervalos')['y_real'].mean()
    prom_pred = df_precision.groupby('Intervalos')['y_predicho'].mean()
    mae_cat = df_precision.groupby('Intervalos').apply(lambda df: metrics.mean_absolute_error(df['y_real'], df['y_predicho']) )
    mape_cat = df_precision.groupby('Intervalos').apply(lambda df: metrics.mean_absolute_percentage_error(df['y_real'], df['y_predicho']) )
    rmspe_cat = df_precision.groupby('Intervalos').apply(lambda df: rmspe(df['y_real'], df['y_predicho']) )

    resultado = pd.DataFrame({
        'Casos': frecuencia,
        'Casos (%)': porcentaje,
        'Prom. real': prom_real,
        'Prom. pred': prom_pred,
        'MAE': mae_cat,
        'MAPE': mape_cat,
        'RMSPE': rmspe_cat
    })

    distribucion = df_precision.groupby(['Intervalos', 'clas_pres']).size().unstack(fill_value=0)

    # Asegurar el orden deseado de las columnas
    for col in ['Subestimacion', 'Coincidencia', 'Sobreestimacion']:
        if col not in distribucion.columns:
            distribucion[col] = 0
    distribucion = distribucion[['Subestimacion', 'Coincidencia', 'Sobreestimacion']]

    distribucion = distribucion.div(distribucion.sum(axis=1), axis=0)

    resultado_final = resultado.join(distribucion, on='Intervalos')

    return resultado_final

def calcular_precision_x_categorias_segmento(y_pred, y_real, df_segmento, umbral):
    # Calcular el porcentaje de precisión
    porc_pres = (y_pred - y_real) / y_real

    # Definir condiciones para clasificar
    condiciones = [
        (porc_pres >= -umbral) & (porc_pres <= umbral),  # Coincidencia
        (porc_pres > umbral),                            # Sobreestimación
        (porc_pres < -umbral)                            # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Clasificar según las condiciones
    clasificacion = np.select(condiciones, clases, default='No clasificado')

    # Crear DataFrame con resultados
    df_precision = pd.DataFrame({
        'y_real': y_real,
        'y_predicho': y_pred,
        'porc_pres': porc_pres,
        'clas_pres': clasificacion,
        'Categoría': df_segmento
    })

    # Calcular frecuencia y promedio por categoría
    frecuencia = df_precision['Categoría'].value_counts().sort_index()
    frecuencia_porc = frecuencia / frecuencia.sum()
    prom_real = df_precision.groupby('Categoría')['y_real'].mean().sort_index()
    prom_pred = df_precision.groupby('Categoría')['y_predicho'].mean().sort_index()

    resultado = pd.DataFrame({
        'Casos': frecuencia,
        'Casos (%)': frecuencia_porc,
        'Prom. real': prom_real,
        'Prom. pred.': prom_pred
    })

    # Distribución por clase de precisión
    distribucion = df_precision.groupby(['Categoría', 'clas_pres']).size().unstack(fill_value=0)

    # Asegurar columnas en orden deseado
    for col in ['Subestimacion', 'Coincidencia', 'Sobreestimacion']:
        if col not in distribucion.columns:
            distribucion[col] = 0
    distribucion = distribucion[['Subestimacion', 'Coincidencia', 'Sobreestimacion']]

    # Calcular porcentajes
    distribucion = distribucion.div(distribucion.sum(axis=1), axis=0)

    # Unir resultados
    resultado_final = resultado.join(distribucion, on='Categoría')

    return resultado_final

def Arbol_Regresion(df,X_names,y_name,porc_min_samples_leaf,omitir_nulos=True,mostrar_prop=True):
    """
    Procesa un Arbol de Regresion.
    
    :param df: DataFrame a procesar.
    :param X_names: Nombre de la/s variable/s independientes.
    :param y_name: Nombre de la variable objetivo.
    :param porc_min_samples_leaf: Porcentaje mínimo del total de las muestras en las hojas.
    :param omitir_nulos: Booleano que indica si se deben omitir los valores nulos.
    :param mostrar_prop: Booleano que indica si se quiere mostrar las muestras como proporcion.
    :return: Grafico del arbol de regresion.
    """
    df_ar = df

    if omitir_nulos:
        df_ar = df_ar.dropna(subset=X_names)
    else:
        df_ar

    X = df_ar[X_names]
    y = df_ar[y_name]

    regressor = DecisionTreeRegressor(random_state = 0,max_depth=5, min_samples_leaf = int(len(X) * porc_min_samples_leaf))
    regressor.fit(X, y)
    # Gráfico
    fig, ax = plt.subplots(figsize=(20,10))

    print(f"Profundidad del árbol: {regressor.get_depth()}")
    print(f"Número de nodos terminales: {regressor.get_n_leaves()}")

    plot = plot_tree(
                decision_tree = regressor,
                feature_names = X.columns,
                class_names   = [y.name],
                proportion    = mostrar_prop,
                filled        = True,
                impurity      = False,
                fontsize      = 10,
                precision     = 8,
                ax            = ax
            )   

def box_plot_outliers(data,list_var_num):
    # Crear subplots para mostrar gráficos al lado de otro con un límite de hasta seis boxplots por fila
    df = data[list_var_num]
    variables = df.columns
    num_vars = len(variables)
    num_cols = min(6, num_vars)
    num_rows = (num_vars + num_cols - 1) // num_cols

    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(18, num_rows * 3.5))

    # Asegurarse de que axes sea un array bidimensional
    axes = np.array(axes).reshape(num_rows, num_cols)

    for i, variable in enumerate(variables):
        row = i // num_cols
        col = i % num_cols
        df.boxplot(column=variable, ax=axes[row, col])
        axes[row, col].set_title(f'{variable}')
        axes[row, col].set_xlabel('Variables')
        axes[row, col].set_ylabel('Valores')

    # Eliminar subplots vacíos si hay menos variables que subplots
    for j in range(i + 1, num_rows * num_cols):
        fig.delaxes(axes.flatten()[j])

    plt.tight_layout()
    plt.show()

def metricas_clasificacion(y_train_real, y_train_predicho, y_test_real, y_test_predicho, plot_confusion_matrix=False):
    # Train
    accuracy_train = metrics.accuracy_score(y_train_real, y_train_predicho)
    auc_train = roc_auc_score(y_train_real, y_train_predicho)
    precision_train = metrics.precision_score(y_train_real, y_train_predicho, average='weighted')
    recall_train = metrics.recall_score(y_train_real, y_train_predicho, average='weighted')
    f1_train = metrics.f1_score(y_train_real, y_train_predicho, average='weighted')
    kappa_train = cohen_kappa_score(y_train_real, y_train_predicho)
    gini_train = 2 * auc_train - 1
    
    # Test
    accuracy_test = metrics.accuracy_score(y_test_real, y_test_predicho)
    auc_test = roc_auc_score(y_test_real, y_test_predicho)
    precision_test = metrics.precision_score(y_test_real, y_test_predicho, average='weighted')
    recall_test = metrics.recall_score(y_test_real, y_test_predicho, average='weighted')
    f1_test = metrics.f1_score(y_test_real, y_test_predicho, average='weighted')
    kappa_test = cohen_kappa_score(y_test_real, y_test_predicho)
    gini_test = 2 * auc_test - 1
    
    # Resumen
    resultados = {
        'Accuracy': [accuracy_train, accuracy_test],
        'AUC': [auc_train, auc_test],
        'Recall': [recall_train, recall_test],
        'Precision': [precision_train, precision_test],
        'F1 Score': [f1_train, f1_test],
        'Kappa': [kappa_train, kappa_test],
        'Gini': [gini_train, gini_test]
    }
    resultados = pd.DataFrame(resultados, index=['Train', 'Test'])
    
    if plot_confusion_matrix:
        fig, axs = plt.subplots(1, 2, figsize=(10, 5))
        
        # Primer gráfico: Matriz de confusión para el conjunto de entrenamiento
        metrics.plot_confusion_matrix(y_train_real, y_train_predicho, ax=axs[0], cmap='Blues')
        axs[0].set_title('Train Confusion Matrix')
        
        # Segundo gráfico: Matriz de confusión para el conjunto de prueba
        metrics.plot_confusion_matrix(y_test_real, y_test_predicho, ax=axs[1], cmap='Blues')
        axs[1].set_title('Test Confusion Matrix')
        
        plt.tight_layout()
    
    return resultados

def calcular_casos_x_deciles(df, variable, puntos_de_corte=None):
    tiene_nulos = df[variable].isnull().any()
    
    # Verificar si la variable es categórica
    if df[variable].dtype == 'object' or df[variable].dtype.name == 'category':
        puntos_de_corte = df[variable].unique().tolist()
        cantidad_por_intervalo = df[variable].value_counts().reset_index()
        cantidad_por_intervalo.columns = [variable, 'Cantidad']
    else:
        # Calcular los puntos de corte para los deciles si no se proporcionan
        if puntos_de_corte is None:
            puntos_de_corte = df[variable].quantile([0.1 * i for i in range(11)]).tolist()
            puntos_de_corte = list(set(puntos_de_corte))
            puntos_de_corte.sort()
        
        # Calcular la cantidad de casos por cada intervalo
        intervalos = pd.cut(df[variable], bins=puntos_de_corte, right=False)
        cantidad_por_intervalo = df.groupby(intervalos).size().reset_index(name='Cantidad')
    
    df_dec = pd.DataFrame({
        'Intervalo': cantidad_por_intervalo[variable].astype(str),
        'Casos': cantidad_por_intervalo['Cantidad'],
        'Casos(%)': cantidad_por_intervalo['Cantidad'] / len(df),
    })
    
    if tiene_nulos:
        cantidad_nulos = df[variable].isnull().sum()
        df_nulos = pd.DataFrame({'Intervalo': ['Nulos'], 'Casos': cantidad_nulos, 'Casos(%)': cantidad_nulos / len(df)})
        df_dec = pd.concat([df_nulos, df_dec], ignore_index=True)
    
    return df_dec

def calcular_psi_x_intervalo(df1, df2, variable, puntos_de_corte=None):
    # Verificar si la variable es categórica
    if df1[variable].dtype == 'object' or df1[variable].dtype.name == 'category':
        puntos_de_corte = df1[variable].unique().tolist()
    else:
        # Calcular los puntos de corte para los deciles si no se proporcionan
        if puntos_de_corte is None:
            puntos_de_corte = df1[variable].quantile([0.1 * i for i in range(11)]).tolist()
            puntos_de_corte = list(set(puntos_de_corte))  # Para eliminar posibles duplicados
            puntos_de_corte.sort()

    prop_1 = calcular_casos_x_deciles(df1, variable, puntos_de_corte)['Casos(%)']
    prop_2 = calcular_casos_x_deciles(df2, variable, puntos_de_corte)['Casos(%)']

    df_psi = pd.DataFrame({
        'Intervalo': calcular_casos_x_deciles(df1, variable, puntos_de_corte)['Intervalo'].astype(str),
        'PSI': (prop_1 - prop_2) * np.log(prop_1 / prop_2)
    })

    suma_psi = df_psi['PSI'].sum()

    total_row = pd.DataFrame({'Intervalo': ['Total'], 'PSI': [suma_psi]})
    df_psi = pd.concat([df_psi, total_row], ignore_index=True)

    return df_psi

def obtener_psi_variables(df1, df2, lista_var):
    lista_psi = []
    for variable in lista_var:
        psi = calcular_psi_x_intervalo(df1, df2, variable)['PSI'].iloc[-1]
        lista_psi.append(psi)

    return pd.DataFrame({'Variable':df1[lista_var].columns,'PSI':lista_psi})

def graficar_psi_variables(df1, df2, lista_var):
    lista_psi = []
    for variable in lista_var:
        psi = calcular_psi_x_intervalo(df1, df2, variable)['PSI'].iloc[-1]
        lista_psi.append(psi)

    psi_df = pd.DataFrame({'Variable':df1[lista_var].columns,'PSI':lista_psi})
    #psi_df['PSI'] = pd.to_numeric(psi_df['PSI'], errors='coerce').fillna(0)

    # Grafico 
    max_psi = psi_df['PSI'].max()
    if np.isfinite(max_psi):
        x_limit = max(0.3, max_psi * 1.1)
    else:
        x_limit = 0.3

    plt.figure(figsize=(10, 6))
    bars = plt.barh(psi_df['Variable'], psi_df['PSI'], color='skyblue')
    plt.axvline(x=0.1, color='orange', linestyle='--', label='Umbral Moderado')
    plt.axvline(x=0.25, color='red', linestyle='--', label='Umbral Alto')
    plt.title('Population Stability Index (PSI) por Variable')
    plt.xlabel('PSI')
    plt.ylabel('Variables')
    plt.legend(loc='upper right')
    plt.xlim(0, x_limit)

    # Agregar etiquetas a las barras
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height() / 2, f'{width:.4f}', ha='center', va='center')

    plt.tight_layout()
    plt.show()
        
    #return lista_psi

def eliminacion_recursiva_de_variables(X, y, model, list_num_features, umbral_coinc):
    
    rfe_0 = RFE(estimator=model, n_features_to_select = max(list_num_features))
    rfe_0.fit(X, y)

    variables_seleccionadas_0 = list(X.columns[rfe_0.support_])

    X = X[variables_seleccionadas_0]

    list_num_features_ord = sorted(list_num_features, reverse=True)
    list_num_features_ord.remove(max(list_num_features))
    lista_var_elim = []

    df_total = []

    for i in tqdm_notebook(list_num_features_ord):

        rfe = RFE(estimator=model, n_features_to_select=i)

        rfe.fit(X, y)

        X_train_selected = rfe.transform(X)

        model.fit(X_train_selected, y)

        y_pred_train = np.exp(model.predict(X_train_selected))

        df_metricas_rfe = metricas_regresion(np.exp(y), y_pred_train,np.exp(y), y_pred_train).loc[['Train']]
        df_coincidencias_rfe = calcular_coincidencia(y_pred_train, y_pred_train, np.exp(y), np.exp(y), umbral_coinc ).loc[['Train']]

        variables_eliminadas = list(X.columns[~rfe.support_])
        ultima_variable_eliminada = list(set(variables_eliminadas) - set(lista_var_elim))
        lista_var_elim.extend(ultima_variable_eliminada)

        df_num_var = pd.DataFrame({"Num_Variables":i,"Var_Eliminada":ultima_variable_eliminada}, index = ['Train'])
        df_concat_metricas = pd.concat([df_num_var,df_metricas_rfe, df_coincidencias_rfe], axis=1)

        df_total.append(df_concat_metricas)
    
    # Concatenando todos los dataframes de la lista
    df_total_df = pd.concat(df_total, axis = 0, ignore_index=True)
    # Invirtiendo el orden de las filas
    df_total_df = df_total_df.iloc[::-1].reset_index(drop=True)

    print(f'Número de variables seleccionadas: {rfe.n_features_}')
    variables_seleccionadas = list(X.columns[rfe.support_])
    print('\nVariables seleccionadas:', variables_seleccionadas)
    display(df_total_df)







### sbp_modelo2

In [ ]:


import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None 
import os
import sys
import time
import pyodbc
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import plotly.express as px
import lightgbm as lgb
from sklearn import tree
from sklearn import metrics
from sklearn.tree import DecisionTreeRegressor
from matplotlib.backends.backend_pdf import PdfPages
from dython.nominal import associations
from sklearn.preprocessing import OneHotEncoder

class MedirTiempoEjecucion:
    def __enter__(self):
        self.start_time = time.time()
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        elapsed_time = time.time() - self.start_time
        print(f"Tiempo transcurrido: {int(elapsed_time // 3600)} horas, {int((elapsed_time % 3600) // 60)} minutos y {elapsed_time % 60:.3f} segundos")
        
#Resumen de las métricas de rendimiento de un modelo de regresión
def metricas_regresion(y_train_real,y_train_predicho,y_test_real,y_test_predicho,plot_actual_vs_predicted=False):
    #Train
    mae_train = metrics.mean_absolute_error(y_train_real, y_train_predicho)
    mape_train = metrics.mean_absolute_percentage_error(y_train_real, y_train_predicho)
    mse_train = metrics.mean_squared_error(y_train_real, y_train_predicho)
    rmse_train = metrics.root_mean_squared_error(y_train_real, y_train_predicho)
    r2_train = metrics.r2_score(y_train_real, y_train_predicho)
    #Test
    mae_test = metrics.mean_absolute_error(y_test_real, y_test_predicho)
    mape_test = metrics.mean_absolute_percentage_error(y_test_real, y_test_predicho)
    mse_test = metrics.mean_squared_error(y_test_real, y_test_predicho)
    rmse_test = metrics.root_mean_squared_error(y_test_real, y_test_predicho)
    r2_test = metrics.r2_score(y_test_real, y_test_predicho)
    #Resumen
    resultados = { 'MAE': [mae_train, mae_test],'MAPE': [mape_train, mape_test], 'MSE': [mse_train, mse_test], 'RMSE': [rmse_train, rmse_test], 'R2': [r2_train, r2_test] }
    resultados = pd.DataFrame(resultados, index=['Train', 'Test'])
    if plot_actual_vs_predicted:
        fig, axs = plt.subplots(1, 2, figsize=(6,3))
        # Primer gráfico:
        display_train = metrics.PredictionErrorDisplay(y_true=y_train_real, y_pred=y_train_predicho)
        display_train.plot(ax=axs[0], kind="actual_vs_predicted")   # actual_vs_predicted / residual_vs_predicted
        axs[0].set_title('Train')
        # Segundo gráfico:
        display_test = metrics.PredictionErrorDisplay(y_true=y_test_real, y_pred=y_test_predicho)
        display_test.plot(ax=axs[1], kind="actual_vs_predicted")    # actual_vs_predicted / residual_vs_predicted
        axs[1].set_title('Test')
        plt.tight_layout()
    return resultados

# Crear matriz de correlación de Spearman de las variables explicativas con el target
def matriz_corr_target(df_variables, target, label_var):
    label_var_1 = list(label_var)
    label_var_1.append(target)
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(2)
    return resultado

# Convertir los valores especiales a valores perdidos
def convertir_valores_especiales_a_missing(df):
    try:
        df = df.replace({-9000001: np.NaN, -9999999: np.NaN, -9000000: np.NaN, -99000632: np.NaN, -99000720: np.NaN, -99000792: np.NaN, -9999998: np.NaN, -9999990: np.NaN,
                        -9999997: np.NaN, -9999996: np.NaN, -9999995: np.NaN, -9999994: np.NaN, -9999993: np.NaN, -9999992: np.NaN, -9999991: np.NaN, pd.NA: np.NaN})
    except:
        df = df.replace({-9000001: np.nan, -9999999: np.nan, -9000000: np.nan, -99000632: np.nan, -99000720: np.nan, -99000792: np.nan, -9999998: np.nan, -9999990: np.nan,
                        -9999997: np.nan, -9999996: np.nan, -9999995: np.nan, -9999994: np.nan, -9999993: np.nan, -9999992: np.nan, -9999991: np.nan, pd.NA: np.nan})

    return df

#Obtiene la lista con las variables constantes
def obtener_var_constantes(df, var_excl=[], umbral_const=0.9999):
    list_var_const = []
    for col in df.columns:
        if col in var_excl:
            continue
        if (df[col].dropna().nunique() == 1) or (df[col].value_counts().iloc[0]/len(df) >= umbral_const):
            list_var_const.append(col)
    return list_var_const

# Crear matriz de correlación de Spearman de las variables explicativas con el target
def matriz_corr_target(df_variables, target, label_var):
    # Filtrar las columnas que existen en el DataFrame
    label_var_1 = [col for col in label_var if col in df_variables.columns]
    label_var_1.append(target)
    
    # Calcular las correlaciones de Spearman
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    
    # Ordenar las columnas por correlación
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    
    # Crear la matriz de correlación
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(2)
    
    return resultado
def matriz_corr_target_v2(df_variables, target, label_var):
    # Filtrar las columnas que existen en el DataFrame
    label_var_1 = [col for col in label_var if col in df_variables.columns]
    label_var_1.append(target)
    
    # Calcular las correlaciones de Spearman
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    
    # Ordenar las columnas por correlación
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    
    # Crear la matriz de correlación
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(2)
    
    # Crear la máscara para la diagonal
    mask = np.zeros_like(resultado, dtype=bool)
    np.fill_diagonal(mask, True)
    
    # Graficar el heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(resultado, annot=True, cmap='Blues', mask=~mask, cbar=False, linewidths=.5, linecolor='black')
    sns.heatmap(resultado, annot=True, cmap='Blues', mask=mask, cbar=True, linewidths=.5, linecolor='black')
    
    return plt.show()

def porcentaje_categorias_plot(data):
    
    category_percentages = {}
    for col in data.columns:
        category_percentages[col] = (data[col].value_counts(normalize=True) * 100).reset_index()
        category_percentages[col].columns = ['Category', 'Percentage']
        category_percentages[col]['Variable'] = col

    category_percentages_df = pd.concat(category_percentages.values())
    
    variables_above_5 = category_percentages_df[category_percentages_df['Percentage'] > 5]['Variable'].unique()
    variables_below_5 = category_percentages_df[category_percentages_df['Percentage'] <= 5]['Variable'].unique()

    category_percentages_df['Order'] = category_percentages_df['Variable'].apply(lambda x: 1 if x in variables_above_5 else 2)

    fig = px.bar(category_percentages_df, y='Variable', x='Percentage', color='Category',
                title='Porcentaje de Categorías por Variable',
                labels={'Percentage': '% de Categoría'},
                category_orders={'Variable': list(variables_above_5) + list(variables_below_5)},
                color_discrete_sequence=px.colors.qualitative.Pastel)

    fig.add_shape(type='line',
                x0=0, y0=len(variables_above_5) - 0.5,
                x1=100, y1=len(variables_above_5) - 0.5,
                line=dict(color='Black', width=3))

    fig.update_layout(barmode='stack', showlegend=False)

    fig.show()

def porcentaje_categorias(data):
    # Lista de variables cualitativas
    variables_with_non_representative_categories = []

    # Bucle para analizar cada variable cualitativa
    for var in data.columns:
        if var in data.columns:  
            freq = data[var].value_counts(normalize=True)  
            dominant_category_percentage = freq.max() 
            non_representative_categories = freq[freq < 0.05]  
    
            if not non_representative_categories.empty:
                num_categories = freq.count()
                num_non_representative_categories = non_representative_categories.count()
                variables_with_non_representative_categories.append((var, num_categories, num_non_representative_categories, 
                                                                    round(dominant_category_percentage*100,2), 
                                                                    round(num_non_representative_categories/num_categories*100,2)))
    
    non_representative = pd.DataFrame(variables_with_non_representative_categories, columns=['Variable', 'Total categorias',
                                                                                            'Total cat. no representativas','% Clase predominante',
                                                                                            '% Cat. no representativas'])
    return non_representative.sort_values(by=['% Clase predominante', '% Cat. no representativas'], ascending=False)

# Muestra las importancias gain y split de un modelo de regresión LightGBM ya entrenado
def importancias_lgbm(modelo,datos):
    split_importance = modelo.feature_importances_
    gain_importance = modelo.booster_.feature_importance(importance_type='gain')

    f_imp = pd.DataFrame({
        'Var': datos.columns,
        'Split_Importance': split_importance,
        'Gain_Importance': gain_importance
    })

    # Calcular el peso de cada importancia
    f_imp['Split_Weight'] = f_imp['Split_Importance'] / f_imp['Split_Importance'].sum() 
    f_imp['Gain_Weight'] = f_imp['Gain_Importance'] / f_imp['Gain_Importance'].sum()

    # Ordenar el DataFrame por la importancia de 'gain'
    return f_imp.sort_values("Gain_Importance", ascending=False,ignore_index=True)

def plot_null_percentage(df):
    """
    Esta función genera una gráfica de barras horizontales que muestra el porcentaje de valores nulos
    por cada variable en un DataFrame dado.
    
    Parámetros:
    df (pandas.DataFrame): El DataFrame para el cual se calculará y visualizará el porcentaje de valores nulos.
    """
    # Calcular el porcentaje de valores nulos por columna
    porcentaje_nulos = df.isnull().mean() * 100
    porcentaje_nulos = porcentaje_nulos.sort_values(ascending=True)

    # Visualizar el porcentaje de valores nulos
    #plt.figure(figsize=(8,6))
    bars = plt.barh(porcentaje_nulos.index, porcentaje_nulos)
    plt.xlabel('Porcentaje')
    plt.title('Porcentaje de valores nulos por cada variable')
    plt.xlim(0, 100)
    
    # Añadir etiquetas con el valor del porcentaje en cada barra
    for bar in bars:
        xval = bar.get_width()
        plt.text(xval + 1, bar.get_y() + bar.get_height()/2, round(xval, 2), va='center', ha='left')
    
    plt.show()
    
def crea_carpeta(ruta):
    dir = os.path.join(ruta)
    if not os.path.exists(dir): os.mkdir(dir)

def extrae_cortes_arbol(df,col,arbol):
    cuts = arbol.tree_.threshold
    cuts = np.unique(cuts)
    cuts = np.insert(cuts,0,-99000720)
    cuts = np.insert(cuts,0,-99000632)
    cuts = np.insert(cuts,0,-9999999)
    cuts = np.insert(cuts,0,-9999998)
    cuts = np.insert(cuts,0,-9999990)
    cuts = np.insert(cuts,0,-9000000)
    cuts = np.insert(cuts,0,np.max(cuts))
    cuts = np.sort(cuts)
    cuts = np.delete(cuts, np.where(cuts == -2))
    return cuts

def extrae_cortes_int(df,col):
    cuts = []
    if df[col].dtype == "int64":
        cuts = df[col].drop_duplicates().to_list()
        cuts = np.unique(cuts)
        cuts = np.insert(cuts,0,-9999999)
        cuts = np.insert(cuts,0,-9999998)
        cuts = np.insert(cuts,0,-9999990)
        cuts = np.insert(cuts,0,-9000000)
        cuts = np.insert(cuts,0,np.max(cuts))
        cuts = np.sort(cuts)
        cuts = np.delete(cuts, np.where(cuts == -2))
    return cuts

def bin_variable_regresion(df,col,cuts,target,estimado=None):
    n = len(cuts)
    variable = []
    orden_cota = []
    cotas = []
    cota_inf = []
    cota_sup = []
    sql = []
    sql_num = []
    media = []
    if estimado is not None: media_est = []
    conteo = []

    df_count = df[col]
    aux = 1
    #print(n)
    for c in range(n):
        #print(c)
        if aux < 100: orden = ("00"+str(aux))[-2:]
        else: orden = str(aux)
        
        if c == 0: #Primer valor
            df_count = df[df[col] <= cuts[c]]
            cotas_c = orden+". (-, "+str(round(cuts[c],3))+"]"
            cota_inf_c = float('-inf')
            cota_sup_c = cuts[c]
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
            sql_n = "when "+col+" <= "+str(cuts[c])+" then "+str(aux)
        elif c == n-1: #Ultimo valor
            df_count = df[df[col] > cuts[c]] 
            cotas_c = orden+". ("+str(round(cuts[c],3))+", +)"
            cota_inf_c = cuts[c]
            cota_sup_c = float('inf')
            sql_c = "when "+col+" > "+str(cuts[c])+" then '"+cotas_c+"' end ["+col+"_cat]"
            sql_n = "when "+col+" > "+str(cuts[c])+" then "+str(aux)+" end ["+col+"_cat]"
        else: #Valores intermedios
            if (cuts[c-1] < cuts[c]) & (cuts[c] == 0):
                df_count = df[(df[col] < cuts[c])&(df[col] > cuts[c-1])] 
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+")"
            elif (cuts[c-1] == cuts[c]) & (cuts[c] == 0):
                df_count = df[df[col] == cuts[c]]
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            else:
                df_count = df[(df[col] <= cuts[c])&(df[col] > cuts[c-1])] 
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
            sql_n = "when "+col+" <= "+str(cuts[c])+" then "+str(aux)
            cota_inf_c = cuts[c-1]
            cota_sup_c = cuts[c]
        
        if "-9000000.0]" in cotas_c: 
            cotas_c = "Missing"
            sql_c = "when "+col+" is null then '"+cotas_c+"'"
            sql_n = "when "+col+" is null then "+str(aux)

        if aux == 1:
            sql_c = ",case " + sql_c
            sql_n = ",case " + sql_n
            
        media_c = df_count[target].mean()
        if estimado is not None: media_est_c = df_count[estimado].mean()
        conteo_c = len(df_count.index)
        
        if conteo_c > 0:
            variable.append(col)
            orden_cota.append(aux)
            cotas.append(cotas_c)
            cota_inf.append(cota_inf_c)
            cota_sup.append(cota_sup_c)
            sql.append(sql_c)
            sql_num.append(sql_n)
            media.append(media_c)
            if estimado is not None: media_est.append(media_est_c)
            conteo.append(conteo_c)
            aux = aux + 1
        
        if conteo_c == 0 and c == n-1:
            sql[-1] += " end ["+col+"_cat]"
    
    media = [round(x,3) for x in media]
    if estimado is not None: media_est = [round(x,3) for x in media_est]
    cota_inf = [round(x,3) for x in cota_inf]
    cota_sup = [round(x,3) for x in cota_sup]
    conteo_pct = [round(x/sum(conteo),3) for x in conteo]
    
    dict_resumen = ""
    dict_resumen = dict_resumen + "'Variable':variable,'Orden_Cota':orden_cota,'Cotas':cotas,'Cota_Inf':cota_inf,'Cota_Sup':cota_sup,'Media':media"
    if estimado is not None: dict_resumen = dict_resumen + ",'Media_Est':media_est"
    dict_resumen = dict_resumen + ",'Total':conteo,'Total_Pct':conteo_pct,'Sql':sql,'Sql_num':sql_num"
    dict_resumen = "{" + dict_resumen + "}"
    dict_resumen = eval(dict_resumen)
    
    df_resumen = pd.DataFrame(dict_resumen)
    
    return df_resumen

def correlacion(df,metodo = "Spearman",target=None):
    
    print("....Correlacion "+metodo)
    cor = df.corr(method=metodo)
    df_cor = pd.DataFrame(cor)
    df_cor_abs = np.abs(df_cor)
    
    #Target
    if target is not None:
        df_cor_target = pd.DataFrame(df_cor_abs[target])
        df_cor_target = df_cor_target.sort_values(by = target,ascending = False)
        df_cor_target = df_cor_target.reset_index()
        df_cor_target = df_cor_target.rename(columns={"index":'Variable'})
        df_cor_target = df_cor_target[df_cor_target['Variable'] != target]
        df_cor_target["Top"] = df_cor_target.index
        
        return df_cor, df_cor_abs, df_cor_target
    else:
        return df_cor, df_cor_abs
    

    from dython.nominal import associations

def correlacion_cat(df,cols,target):
    
    print("....Correlacion Categoricas")
    assoc = associations(df, 
                    nominal_columns=cols,
                    numerical_columns=target,
                    plot=False,num_num_assoc='spearman',nom_num_assoc='correlation_ratio',nom_nom_assoc='cramer'
                    ,cmap='coolwarm',title='Correlación entre las Variables Nominales y el Target',figsize=(9,6))
    
    return assoc['corr']
    
def correlacion_target(df,metodo,target):
    
    print("....Correlacion "+metodo)
    cor = df.corrwith(df[target],method=metodo)
    df_cor_target = pd.DataFrame(cor)
    df_cor_target = np.abs(df_cor_target)
    df_cor_target = df_cor_target.rename(columns={0:target})
    df_cor_target = df_cor_target.sort_values(by = target,ascending = False)
    df_cor_target = df_cor_target.reset_index()
    df_cor_target = df_cor_target.rename(columns={"index":'Variable'})
    df_cor_target = df_cor_target[df_cor_target['Variable'] != target]
    df_cor_target["Top"] = df_cor_target.index

    return df_cor_target
    
def descripcion_variable(df,col,redondeo=None):
    
    if redondeo is None: redondeo = 3
    
    # Medidas
    describe = df[col].describe()
    
    if df[col].dtype in (["int64","float64"]):
        iqr = describe["75%"] - describe["25%"]
        outlo = describe["25%"] - 1.5*iqr
        outhi = describe["75%"] + 1.5*iqr
        dict_descripcion = {"Variable":col
                                   ,"Type":df[col].dtype
                                   ,"Freq":df.shape[0]
                                   ,"Unico":len(df[col].unique())
                                   ,"Media":describe["mean"]
                                   ,"Desv":describe["std"]
                                   ,"Min":describe["min"]
                                   ,"Q1":describe["25%"]
                                   ,"Mediana":describe["50%"]
                                   ,"Q3":describe["75%"]
                                   ,"Max":describe["max"]
                                   ,"Miss":df[col].isna().sum()
                                   ,"Cero":df[col].eq(0).sum()
                                   ,"Pos":df[col].gt(0).sum()
                                   ,"Neg":df[col].lt(0).sum()
                                   ,"OutLo":df[col].lt(outlo).sum()
                                   ,"OutHi":df[col].gt(outhi).sum()
                                   ,"pMiss":round(df[col].isna().sum() / df.shape[0],redondeo)
                                   ,"pCero":round(df[col].eq(0).sum() / df.shape[0],redondeo)
                                   ,"pPos":round(df[col].gt(0).sum() / df.shape[0],redondeo)
                                   ,"pNeg":round(df[col].lt(0).sum() / df.shape[0],redondeo)
                                   ,"pOutLo":round(df[col].lt(outlo).sum() / df.shape[0],redondeo)
                                   ,"pOutHi":round(df[col].gt(outhi).sum() / df.shape[0],redondeo)
                                   ,"Moda":""
                                   ,"nModa":0
                                   ,"pModa":0.0}
    else:
        dict_descripcion = {"Variable":col
                                   ,"Type":df[col].dtype
                                   ,"Freq":df.shape[0]
                                   ,"Unico":len(df[col].unique())
                                   ,"Media":0
                                   ,"Desv":0
                                   ,"Min":0
                                   ,"Q1":0
                                   ,"Mediana":0
                                   ,"Q3":0
                                   ,"Max":0
                                   ,"Miss":df[col].isna().sum()
                                   ,"Cero":0
                                   ,"Pos":0
                                   ,"Neg":0
                                   ,"OutLo":0
                                   ,"OutHi":0
                                   ,"pMiss":round(df[col].isna().sum() / df.shape[0],redondeo)
                                   ,"pCero":0
                                   ,"pPos":0
                                   ,"pNeg":0
                                   ,"pOutLo":0
                                   ,"pOutHi":0
                                   ,"Moda":describe["top"]
                                   ,"nModa":describe["freq"]
                                   ,"pModa":round(describe["freq"] / df.shape[0],redondeo)}
    
    df_descripcion = pd.DataFrame([dict_descripcion])
    
    return df_descripcion

def bucle_descripcion_variables(df,cols,redondeo=None):
    for i in range(len(cols)):
        df_descripcion = descripcion_variable(df,cols[i],redondeo)
        if i == 0:
            df_descripciones = df_descripcion
        else:
            df_descripciones = pd.concat([df_descripciones,df_descripcion])
    
    return df_descripciones

def eda(df,cols,target,name=None):
    df_target = df[target]
    n = df.shape[0]
    columnas_numeric = df.select_dtypes(exclude='category').columns.to_list()
    columnas_category = df.select_dtypes(include='category').columns.to_list()
    
    #df_total = pd.DataFrame(columns=["Orden","Variable","Type","Freq","Unico","Media","Desv","Min","Q1","Mediana","Q3","Max","Miss","Cero","Pos","Neg","OutLo","OutHi","pMiss","pCero","pPos","pNeg","pOutLo","pOutHi","Moda","nModa","pModa"])
    df_variables = pd.DataFrame(columns=["Variable","Orden_Cota","Cotas","Sql","Media","Total","Total_Pct"])
    
    crea_carpeta(".//Exploracion_Variables")
    if name is None:  
        ruta_pdf = ".//Exploracion_Variables//Resultados.pdf"
    else:
        ruta_pdf = ".//Exploracion_Variables//Resultados_"+name+".pdf"
        
    df_descripciones = bucle_descripcion_variables(df,cols)
    
    with PdfPages(ruta_pdf) as pdf:
        for i in range(len(cols)):
            print("...."+str(i+1)+". "+cols[i])

            # Resumen Total
            #df_total = pd.concat([df_total,df_descripcion])

            if cols[i] in columnas_numeric:

                try:
                    # Base inicial
                    df_X = pd.DataFrame(df[cols[i]])
                    df_X.loc[df_X[cols[i]].isna(),cols[i]] = -9000000
                    df.loc[df[cols[i]].isna(),cols[i]] = -9000000

                    # Arbol
                    arbol = DecisionTreeRegressor(min_samples_split = 0.20,min_samples_leaf = int(n*0.05))#,max_depth = 3
                    arbol.fit(df_X,df_target)

                    # Cortes del Arbol
                    if df[cols[i]].dtype == "int64":
                        cuts = extrae_cortes_int(df,cols[i])
                    else:
                        cuts = extrae_cortes_arbol(df,cols[i],arbol)
                    
                    # Resumen de Variables
                    df_bin_variable = bin_variable_regresion(df,cols[i],cuts,target)
                    df_variables = pd.concat([df_variables,df_bin_variable])

                    # Graficos
                    # Grafico de Barras
                    plt.figure(figsize=(15, 6), dpi=100)

                    plt.subplot(121)
                    plt.bar(df_bin_variable["Orden_Cota"],df_bin_variable["Total"],width=.5)
                    plt.xticks(df_bin_variable["Orden_Cota"])
                    plt.title(str(i+1)+". "+cols[i])
                    axes2 = plt.twinx()
                    axes2.plot(df_bin_variable["Orden_Cota"], df_bin_variable['Media'], color='Orange',marker = "o")

                    # Tabla resumen
                    plt.subplot(122)
                    cell_text = []
                    df_X = df_bin_variable[["Cotas", "Media", "Total","Total_Pct"]]
                    df_X.loc[:,"Media"] = df_X["Media"].apply(lambda x: round(x,3))
                    df_X.loc[:,"Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    #df_X["Media"] = df_X["Media"].apply(lambda x: round(x,1))
                    #df_X["Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    for row in range(len(df_X)):
                        cell_text.append(df_X.iloc[row])

                    tab = plt.table(cellText=cell_text, colLabels=df_X.columns, cellLoc='center',loc='center',colWidths=[0.6, 0.2,0.2,0.2])
                    tab.auto_set_font_size(False)
                    #tab.set_fontsize(12)
                    #tab.scale(2, 2)
                    plt.axis('off')
                    plt.tight_layout()
                    plt.subplots_adjust(left=0.1,bottom=0.1,right=0.9,top=0.9,wspace=0.25,hspace=0.4)

                    pdf.savefig()
                    plt.close()

                    # Arbol
                    fig = plt.figure(figsize=(25,20))
                    _ = tree.plot_tree(arbol, 
                                    feature_names=[cols[i]],
                                    #class_names=iris.target_names,
                                    filled=True)
                    pdf.savefig()
                    plt.close()
                except Exception as e:
                    print("...."+str(i+1)+". Error: ",e)
        #             # Violin
        #             sb.catplot(x=cols[i], y="ingreso_cascada", kind="violin", data=df)
        #             pdf.savefig()
        #             plt.close()

        #             # Cajas
        #             sb.catplot(x=cols[i], y=target, kind="box", data=df)
        #             pdf.savefig()
        #             plt.close()
            if cols[i] in columnas_category:
                try:
                    # Arbol


                    # Tabla Resumen
                    plt.subplot(122)
                    cell_text = []
                    df_X = df_bin_variable[["Cotas", "Media", "Total","Total_Pct"]]
                    df_X.loc[:,"Media"] = df_X["Media"].apply(lambda x: round(x,3))
                    df_X.loc[:,"Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    #df_X["Media"] = df_X["Media"].apply(lambda x: round(x,1))
                    #df_X["Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    for row in range(len(df_X)):
                        cell_text.append(df_X.iloc[row])

                    tab = plt.table(cellText=cell_text, colLabels=df_X.columns, cellLoc='center',loc='center',colWidths=[0.6, 0.2,0.2,0.2])
                    tab.auto_set_font_size(False)
                    #tab.set_fontsize(12)
                    #tab.scale(2, 2)
                    plt.axis('off')
                    plt.tight_layout()
                except Exception as e:
                    print("...."+str(i+1)+". Error: ",e)


    df_variables = df_variables.reset_index()
    
    return df_descripciones, df_variables

def format_elapsed_time(elapsed_time):
    
    h = 0
    m = 0
    s = 0
    time_text = ""
    
    if elapsed_time >= 3600:
        h = int(elapsed_time/3600)
        time_text += str(h)+"h "
        elapsed_time = elapsed_time - h*3600

    if elapsed_time >= 60:
        m = int(elapsed_time/60)
        time_text += str(m)+"m "
        elapsed_time = elapsed_time - m*60
    
    if elapsed_time > 0:
        s = int(elapsed_time)
        time_text += str(s)+"s "
    
    time_text = time_text[0:-1]
    
    return time_text

### sbp_modelo_v2

In [ ]:

import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None 
import os
import sys
import time
#import pyodbc
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import plotly.express as px
import lightgbm as lgb
from sklearn import tree
from sklearn import metrics
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier, plot_tree
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_auc_score, cohen_kappa_score
from sklearn.feature_selection import RFE
from IPython.display import display
from tqdm import tqdm_notebook
from sklearn.base import clone

class MedirTiempoEjecucion:
    def __enter__(self):
        self.start_time = time.time()
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        elapsed_time = time.time() - self.start_time
        print(f"Tiempo transcurrido: {int(elapsed_time // 3600)} horas, {int((elapsed_time % 3600) // 60)} minutos y {elapsed_time % 60:.3f} segundos")
        
#Resumen de las métricas de rendimiento de un modelo de regresión
def metricas_regresion(y_train_real,y_train_predicho,y_test_real,y_test_predicho,plot_actual_vs_predicted=False):
    
    def rmspe(y_true, y_pred):
        return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

    #Train
    mae_train = metrics.mean_absolute_error(y_train_real, y_train_predicho)
    mape_train = metrics.mean_absolute_percentage_error(y_train_real, y_train_predicho)
    mse_train = metrics.mean_squared_error(y_train_real, y_train_predicho)
    rmse_train = metrics.root_mean_squared_error(y_train_real, y_train_predicho)
    rmspe_train = rmspe(y_train_real, y_train_predicho)
    r2_train = metrics.r2_score(y_train_real, y_train_predicho)

    #Test
    mae_test = metrics.mean_absolute_error(y_test_real, y_test_predicho)
    mape_test = metrics.mean_absolute_percentage_error(y_test_real, y_test_predicho)
    mse_test = metrics.mean_squared_error(y_test_real, y_test_predicho)
    rmse_test = metrics.root_mean_squared_error(y_test_real, y_test_predicho)
    rmspe_test = rmspe(y_test_real, y_test_predicho)
    r2_test = metrics.r2_score(y_test_real, y_test_predicho)
    #Resumen
    resultados = { 'MAE': [mae_train, mae_test],'MAPE': [mape_train, mape_test], 'MSE': [mse_train, mse_test], 'RMSE': [rmse_train, rmse_test], 'RMSPE': [rmspe_train, rmspe_test], 'R2': [r2_train, r2_test] }
    resultados = pd.DataFrame(resultados, index=['Train', 'Test'])
    if plot_actual_vs_predicted:
        fig, axs = plt.subplots(1, 2, figsize=(6,3))
        # Primer gráfico:
        display_train = metrics.PredictionErrorDisplay(y_true=y_train_real, y_pred=y_train_predicho)
        display_train.plot(ax=axs[0], kind="actual_vs_predicted")   # actual_vs_predicted / residual_vs_predicted
        axs[0].set_title('Train')
        # Segundo gráfico:
        display_test = metrics.PredictionErrorDisplay(y_true=y_test_real, y_pred=y_test_predicho)
        display_test.plot(ax=axs[1], kind="actual_vs_predicted")    # actual_vs_predicted / residual_vs_predicted
        axs[1].set_title('Test')
        plt.tight_layout()
    return resultados

def metricas_regresion_v2(y_train_real,y_train_predicho,y_test_real,y_test_predicho,y_real_oot1, y_pred_oot1,y_real_oot2, y_pred_oot2,y_real_oot3, y_pred_oot3):

    def rmspe_metric(y_true, y_pred):
        return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

    #   def r2_adj(y_true, y_pred, p):
    #       n = len(y_true)  # asumimos n > p + 1
    #       R2 = metrics.r2_score(y_true, y_pred)
    #       return 1 - (1 - R2) * (n - 1) / (n - p - 1)

    # Mapeo de datasets en el orden requerido
    datasets = [
        ('Train', y_train_real, y_train_predicho),
        ('Test', y_test_real, y_test_predicho),
        ('OOT1', y_real_oot1, y_pred_oot1),
        ('OOT2', y_real_oot2, y_pred_oot2),
        ('OOT3', y_real_oot3, y_pred_oot3),
    ]

    # Calcula métricas en un bucle
    filas = []
    indices = []
    for nombre, y_true, y_pred in datasets:
        mae = metrics.mean_absolute_error(y_true, y_pred)
        mape = metrics.mean_absolute_percentage_error(y_true, y_pred)
        mse = metrics.mean_squared_error(y_true, y_pred)
        rmse = metrics.root_mean_squared_error(y_true, y_pred)
        rmspe = rmspe_metric(y_true, y_pred)
        r2 = metrics.r2_score(y_true, y_pred)
        #   r2aj = r2_adj(y_true, y_pred, p)

        filas.append({
            'MAE': mae,
            'MAPE': mape,
            'MSE': mse,
            'RMSE': rmse,
            'RMSPE': rmspe,
            'R2': r2,
            #   'R2_adj': r2aj
        })
        indices.append(nombre)

    resultados = pd.DataFrame(filas, index=indices)

    return resultados

# Crear matriz de correlación de Spearman de las variables explicativas con el target
def matriz_corr_target(df_variables, target, label_var):
    label_var_1 = list(label_var)
    label_var_1.append(target)
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(3)
    return resultado

# Convertir los valores especiales a valores perdidos
def convertir_valores_especiales_a_missing(df):
    try:
        df = df.replace({-9000001: np.NaN, -9999999: np.NaN, -9000000: np.NaN, -99000632: np.NaN, -99000720: np.NaN, -99000792: np.NaN, -9999998: np.NaN, -9999990: np.NaN,
                        -9999997: np.NaN, -9999996: np.NaN, -9999995: np.NaN, -9999994: np.NaN, -9999993: np.NaN, -9999992: np.NaN, -9999991: np.NaN, -99000800: np.NaN,
                        pd.NA: np.NaN})
    except:
        df = df.replace({-9000001: np.nan, -9999999: np.nan, -9000000: np.nan, -99000632: np.nan, -99000720: np.nan, -99000792: np.nan, -9999998: np.nan, -9999990: np.nan,
                        -9999997: np.nan, -9999996: np.nan, -9999995: np.nan, -9999994: np.nan, -9999993: np.nan, -9999992: np.nan, -9999991: np.nan, -99000800: np.nan,
                        pd.NA: np.nan})

    return df

#Obtiene la lista con las variables constantes
def obtener_var_constantes(df, var_excl=[], umbral_const=0.9999):
    list_var_const = []
    for col in df.columns:
        if col in var_excl:
            continue
        if (df[col].dropna().nunique() == 1) or (df[col].value_counts().iloc[0]/len(df) >= umbral_const):
            list_var_const.append(col)
    return list_var_const

# Crear matriz de correlación de Spearman de las variables explicativas con el target
def matriz_corr_target(df_variables, target, label_var):
    # Filtrar las columnas que existen en el DataFrame
    label_var_1 = [col for col in label_var if col in df_variables.columns]
    label_var_1.append(target)
    
    # Calcular las correlaciones de Spearman
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    
    # Ordenar las columnas por correlación
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    
    # Crear la matriz de correlación
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(3)
    
    return resultado
def matriz_corr_target_v2(df_variables, target, label_var):
    # Filtrar las columnas que existen en el DataFrame
    label_var_1 = [col for col in label_var if col in df_variables.columns]
    label_var_1.append(target)
    
    # Calcular las correlaciones de Spearman
    correlaciones = df_variables[label_var_1].corr(method="spearman")[target].drop(target)
    correlaciones = correlaciones.abs().sort_values(ascending=False)
    
    # Ordenar las columnas por correlación
    lista_ord_corr = correlaciones.index.tolist()
    lista_ord_corr.insert(0, target)
    
    # Crear la matriz de correlación
    resultado = np.abs(df_variables[lista_ord_corr].corr(method='spearman'))
    resultado = resultado.round(2)
    
    # Crear la máscara para la diagonal
    mask = np.zeros_like(resultado, dtype=bool)
    np.fill_diagonal(mask, True)
    
    # Graficar el heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(resultado, annot=True, cmap='Blues', mask=~mask, cbar=False, linewidths=.5, linecolor='black')
    sns.heatmap(resultado, annot=True, cmap='Blues', mask=mask, cbar=True, linewidths=.5, linecolor='black')
    
    return plt.show()

def porcentaje_categorias_plot(data):
    
    category_percentages = {}
    for col in data.columns:
        category_percentages[col] = (data[col].value_counts(normalize=True) * 100).reset_index()
        category_percentages[col].columns = ['Category', 'Percentage']
        category_percentages[col]['Variable'] = col

    category_percentages_df = pd.concat(category_percentages.values())
    
    variables_above_5 = category_percentages_df[category_percentages_df['Percentage'] > 5]['Variable'].unique()
    variables_below_5 = category_percentages_df[category_percentages_df['Percentage'] <= 5]['Variable'].unique()

    category_percentages_df['Order'] = category_percentages_df['Variable'].apply(lambda x: 1 if x in variables_above_5 else 2)

    fig = px.bar(category_percentages_df, y='Variable', x='Percentage', color='Category',
                title='Porcentaje de Categorías por Variable',
                labels={'Percentage': '% de Categoría'},
                category_orders={'Variable': list(variables_above_5) + list(variables_below_5)},
                color_discrete_sequence=px.colors.qualitative.Pastel)

    fig.add_shape(type='line',
                x0=0, y0=len(variables_above_5) - 0.5,
                x1=100, y1=len(variables_above_5) - 0.5,
                line=dict(color='Black', width=3))

    fig.update_layout(barmode='stack', showlegend=False)

    fig.show()

def porcentaje_categorias(data):
    # Lista de variables cualitativas
    variables_with_non_representative_categories = []

    # Bucle para analizar cada variable cualitativa
    for var in data.columns:
        if var in data.columns:  
            freq = data[var].value_counts(normalize=True)  
            dominant_category_percentage = freq.max() 
            non_representative_categories = freq[freq < 0.05]  
    
            if not non_representative_categories.empty:
                num_categories = freq.count()
                num_non_representative_categories = non_representative_categories.count()
                variables_with_non_representative_categories.append((var, num_categories, num_non_representative_categories, 
                                                                    round(dominant_category_percentage*100,2), 
                                                                    round(num_non_representative_categories/num_categories*100,2)))
    
    non_representative = pd.DataFrame(variables_with_non_representative_categories, columns=['Variable', 'Total categorias',
                                                                                            'Total cat. no representativas','% Clase predominante',
                                                                                            '% Cat. no representativas'])
    return non_representative.sort_values(by=['% Clase predominante', '% Cat. no representativas'], ascending=False)

# Muestra las importancias gain y split de un modelo de regresión LightGBM ya entrenado
def importancias_lgbm(modelo,datos):
    split_importance = modelo.feature_importances_
    gain_importance = modelo.booster_.feature_importance(importance_type='gain')

    f_imp = pd.DataFrame({
        'Var': datos.columns,
        'Split_Importance': split_importance,
        'Gain_Importance': gain_importance
    })

    # Calcular el peso de cada importancia
    f_imp['Split_Weight'] = f_imp['Split_Importance'] / f_imp['Split_Importance'].sum() 
    f_imp['Gain_Weight'] = f_imp['Gain_Importance'] / f_imp['Gain_Importance'].sum()
    f_imp['Avg_Weight'] = ( f_imp['Gain_Weight'] + f_imp['Split_Weight'] )/2

    # Ordenar el DataFrame por la importancia de 'gain'
    return f_imp.sort_values("Avg_Weight", ascending=False,ignore_index=True)

def importancias_lgbm_v2(modelo,datos):
    split_importance = modelo.feature_importances_
    gain_importance = modelo.booster_.feature_importance(importance_type='gain')

    f_imp = pd.DataFrame({
        'Var': datos.columns,
        'Split_Importance': split_importance,
        'Gain_Importance': gain_importance
    })

    # Calcular el peso de cada importancia
    f_imp['Split_Weight'] = f_imp['Split_Importance'] / f_imp['Split_Importance'].sum() 
    f_imp['Gain_Weight'] = f_imp['Gain_Importance'] / f_imp['Gain_Importance'].sum()
    f_imp['Avg_Weight'] = ( f_imp['Gain_Weight'] + f_imp['Split_Weight'] )/2
    f_imp['H_Weight'] = 2*f_imp['Gain_Weight']*f_imp['Split_Weight']/( f_imp['Gain_Weight'] + f_imp['Split_Weight'] )
    f_imp['H_Avg_Weight'] = f_imp['H_Weight'] / f_imp['H_Weight'].sum()
    f_imp = f_imp.drop(columns=['H_Weight'])

    # Ordenar el DataFrame por la importancia de 'gain'
    return f_imp.sort_values("H_Avg_Weight", ascending=False,ignore_index=True)

def plot_null_percentage(df):
    """
    Esta función genera una gráfica de barras horizontales que muestra el porcentaje de valores nulos
    por cada variable en un DataFrame dado.
    
    Parámetros:
    df (pandas.DataFrame): El DataFrame para el cual se calculará y visualizará el porcentaje de valores nulos.
    """
    # Calcular el porcentaje de valores nulos por columna
    porcentaje_nulos = df.isnull().mean() * 100
    porcentaje_nulos = porcentaje_nulos.sort_values(ascending=True)

    # Visualizar el porcentaje de valores nulos
    #plt.figure(figsize=(8,6))
    bars = plt.barh(porcentaje_nulos.index, porcentaje_nulos)
    plt.xlabel('Porcentaje')
    plt.title('Porcentaje de valores nulos por cada variable')
    plt.xlim(0, 100)
    
    # Añadir etiquetas con el valor del porcentaje en cada barra
    for bar in bars:
        xval = bar.get_width()
        plt.text(xval + 1, bar.get_y() + bar.get_height()/2, round(xval, 2), va='center', ha='left')
    
    plt.show()
    
def correlacion(df,metodo = "spearman",target=None):
    
    print(".Correlacion "+metodo)
    cor = df.corr(method=metodo)
    df_cor = pd.DataFrame(cor)
    df_cor_abs = np.abs(df_cor)
    
    #Target
    if target is not None:
        df_cor_target = pd.DataFrame(df_cor_abs[target])
        df_cor_target = df_cor_target.sort_values(by = target,ascending = False)
        df_cor_target = df_cor_target.reset_index()
        df_cor_target = df_cor_target.rename(columns={"index":'Variable'})
        df_cor_target = df_cor_target[df_cor_target['Variable'] != target]
        df_cor_target["Top"] = df_cor_target.index
        
        return df_cor, df_cor_abs, df_cor_target
    else:
        return df_cor, df_cor_abs    
def extrae_cortes_arbol(df,cols,arbol):
    if df[cols].dtype == "int64":
        cuts = df[cols].drop_duplicates().to_list()
    else:
        cuts = arbol.tree_.threshold
    cuts = np.unique(cuts)
    #print(cuts)
    #cuts = np.insert(cuts,0,0)
    #cuts = np.insert(cuts,0,0)
    cuts = np.insert(cuts,0,-9999999)
    cuts = np.insert(cuts,0,-9999998)
    cuts = np.insert(cuts,0,-9999990)
    cuts = np.insert(cuts,0,-9000000)
    cuts = np.insert(cuts,0,np.max(cuts))
    cuts = np.sort(cuts)
    cuts = np.delete(cuts, np.where(cuts == -2))
    return cuts
def bin_variable_regresion(df,col,cuts,target,estimado=None):
    n = len(cuts)
    variable = []
    orden_cota = []
    cotas = []
    cota_inf = []
    cota_sup = []
    sql = []
    sql_num = []
    media = []
    if estimado is not None: media_est = []
    conteo = []

    df_count = df[col]
    aux = 1
    #print(n)
    for c in range(n):
        #print(c)
        if aux < 100: orden = ("00"+str(aux))[-2:]
        else: orden = str(aux)
        
        if c == 0: #Primer valor
            df_count = df[df[col] <= cuts[c]]
            cotas_c = orden+". (-, "+str(round(cuts[c],3))+"]"
            cota_inf_c = float('-inf')
            cota_sup_c = cuts[c]
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
            sql_n = "when "+col+" <= "+str(cuts[c])+" then "+str(aux)
        elif c == n-1: #Ultimo valor
            df_count = df[df[col] > cuts[c]] 
            cotas_c = orden+". ("+str(round(cuts[c],3))+", +)"
            cota_inf_c = cuts[c]
            cota_sup_c = float('inf')
            sql_c = "when "+col+" > "+str(cuts[c])+" then '"+cotas_c+"' end ["+col+"_cat]"
            sql_n = "when "+col+" > "+str(cuts[c])+" then "+str(aux)+" end ["+col+"_cat]"
        else: #Valores intermedios
            if (cuts[c-1] < cuts[c]) & (cuts[c] == 0):
                df_count = df[(df[col] < cuts[c])&(df[col] > cuts[c-1])] 
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+")"
            elif (cuts[c-1] == cuts[c]) & (cuts[c] == 0):
                df_count = df[df[col] == cuts[c]]
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            else:
                df_count = df[(df[col] <= cuts[c])&(df[col] > cuts[c-1])] 
                cotas_c = orden+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
            sql_n = "when "+col+" <= "+str(cuts[c])+" then "+str(aux)
            cota_inf_c = cuts[c-1]
            cota_sup_c = cuts[c]
        
        if "-9000000.0]" in cotas_c: 
            cotas_c = "Missing"
            sql_c = "when "+col+" is null then '"+cotas_c+"'"
            sql_n = "when "+col+" is null then "+str(aux)

        if aux == 1:
            sql_c = ",case " + sql_c
            sql_n = ",case " + sql_n
            
        media_c = df_count[target].mean()
        if estimado is not None: media_est_c = df_count[estimado].mean()
        conteo_c = len(df_count.index)
        
        if conteo_c > 0:
            variable.append(col)
            orden_cota.append(aux)
            cotas.append(cotas_c)
            cota_inf.append(cota_inf_c)
            cota_sup.append(cota_sup_c)
            sql.append(sql_c)
            sql_num.append(sql_n)
            media.append(media_c)
            if estimado is not None: media_est.append(media_est_c)
            conteo.append(conteo_c)
            aux = aux + 1
        
        if conteo_c == 0 and c == n-1:
            sql[-1] += " end ["+col+"_cat]"
    
    media = [round(x,3) for x in media]
    if estimado is not None: media_est = [round(x,3) for x in media_est]
    cota_inf = [round(x,3) for x in cota_inf]
    cota_sup = [round(x,3) for x in cota_sup]
    conteo_pct = [round(x/sum(conteo),3) for x in conteo]
    
    dict_resumen = ""
    dict_resumen = dict_resumen + "'Variable':variable,'Orden_Cota':orden_cota,'Cotas':cotas,'Cota_Inf':cota_inf,'Cota_Sup':cota_sup,'Media':media"
    if estimado is not None: dict_resumen = dict_resumen + ",'Media_Est':media_est"
    dict_resumen = dict_resumen + ",'Total':conteo,'Total_Pct':conteo_pct,'Sql':sql,'Sql_num':sql_num"
    dict_resumen = "{" + dict_resumen + "}"
    dict_resumen = eval(dict_resumen)
    
    df_resumen = pd.DataFrame(dict_resumen)
    
    return df_resumen
def bin_variable(cuts,df_subset,target,col,orden):
    n = len(cuts)
    variable = []
    orden_var = []
    orden_cota = []
    cotas = []
    sql = []
    media = []
    conteo = []

    df_count = df_subset[col]
    aux = 1
    #print(n)
    for c in range(n):
        #print(c)
        if c == 0: #Primer valor
            df_count = df_subset[df_subset[col] <= cuts[c]]
            cotas_c = str(aux)+". (-, "+str(round(cuts[c],3))+"]"
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
        elif c == n-1: #Ultimo valor
            df_count = df_subset[df_subset[col] > cuts[c]] 
            cotas_c = str(aux)+". ("+str(round(cuts[c],3))+", +)"
            sql_c = "when "+col+" > "+str(cuts[c])+" then '"+cotas_c+"' end ["+col+"_cat]"
        else: #Valores intermedios
            if (cuts[c-1] < cuts[c]) & (cuts[c] == 0):
                df_count = df_subset[(df_subset[col] < cuts[c])&(df_subset[col] > cuts[c-1])] 
                cotas_c = str(aux)+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+")"
            elif (cuts[c-1] == cuts[c]) & (cuts[c] == 0):
                df_count = df_subset[df_subset[col] == cuts[c]]
                cotas_c = str(aux)+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            else:
                df_count = df_subset[(df_subset[col] <= cuts[c])&(df_subset[col] > cuts[c-1])] 
                cotas_c = str(aux)+". ("+str(round(cuts[c-1],3))+", "+str(round(cuts[c],3))+"]"
            sql_c = "when "+col+" <= "+str(cuts[c])+" then '"+cotas_c+"'"
        
        if "-9000000.0]" in cotas_c: 
            cotas_c = cotas_c[:cotas_c.find(".")+1]+" Missing"
            sql_c = "when "+col+" is null then '"+cotas_c+"'"
        
        if aux == 1: sql_c = ",case " + sql_c
        media_c = df_count[target].mean()
        conteo_c = len(df_count.index)
        
        if conteo_c > 0:
            variable.append(col)
            orden_cota.append(aux)
            orden_var.append(orden)
            cotas.append(cotas_c)
            sql.append(sql_c)
            media.append(media_c)
            conteo.append(conteo_c)
            aux = aux + 1
        
        if conteo_c == 0 and c == n-1:
            sql[-1] += " end ["+col+"_cat]"
    
    df_resumen = pd.DataFrame({"Orden":orden_var,"Variable":variable,"Orden_Cota":orden_cota,"Cotas":cotas,"Sql":sql,"Media":media,"Total":conteo})
    df_resumen["Total_Pct"] = (df_resumen["Total"] / df_resumen["Total"].sum()).round(3)
    
    return df_resumen
def descripcion_variable(df,col,orden,redondeo=None):
    
    if redondeo is None: redondeo = 3
    
    # Medidas
    describe = df[col].describe()
    
    if df[col].dtype in (["int64","float64","int32"]):
        iqr = describe["75%"] - describe["25%"]
        outlo = describe["25%"] - 1.5*iqr
        outhi = describe["75%"] + 1.5*iqr
        dict_descripcion = {"Orden":orden
                                    ,"Variable":col
                                    ,"Type":df[col].dtype
                                    ,"Freq":df.shape[0]
                                    ,"Unico":len(df[col].unique())
                                    ,"Media":describe["mean"]
                                    ,"Desv":describe["std"]
                                    ,"Min":describe["min"]
                                    ,"Q1":describe["25%"]
                                    ,"Mediana":describe["50%"]
                                    ,"Q3":describe["75%"]
                                    ,"Max":describe["max"]
                                    ,"Miss":df[col].isna().sum()
                                    ,"Cero":df[col].eq(0).sum()
                                    ,"Pos":df[col].gt(0).sum()
                                    ,"Neg":df[col].lt(0).sum()
                                    ,"OutLo":df[col].lt(outlo).sum()
                                    ,"OutHi":df[col].gt(outhi).sum()
                                    ,"pMiss":(df[col].isna().sum() / df.shape[0]).round(redondeo)
                                    ,"pCero":(df[col].eq(0).sum() / df.shape[0]).round(redondeo)
                                    ,"pPos":(df[col].gt(0).sum() / df.shape[0]).round(redondeo)
                                    ,"pNeg":(df[col].lt(0).sum() / df.shape[0]).round(redondeo)
                                    ,"pOutLo":(df[col].lt(outlo).sum() / df.shape[0]).round(redondeo)
                                    ,"pOutHi":(df[col].gt(outhi).sum() / df.shape[0]).round(redondeo)
                                    ,"Moda":""
                                    ,"nModa":0
                                    ,"pModa":0.0}
    else:
        dict_descripcion = {"Orden":orden
                                    ,"Variable":col
                                    ,"Type":df[col].dtype
                                    ,"Freq":df.shape[0]
                                    ,"Unico":len(df[col].unique())
                                    ,"Media":0
                                    ,"Desv":0
                                    ,"Min":0
                                    ,"Q1":0
                                    ,"Mediana":0
                                    ,"Q3":0
                                    ,"Max":0
                                    ,"Miss":df[col].isna().sum()
                                    ,"Cero":0
                                    ,"Pos":0
                                    ,"Neg":0
                                    ,"OutLo":0
                                    ,"OutHi":0
                                    ,"pMiss":(df[col].isna().sum() / df.shape[0]).round(redondeo)
                                    ,"pCero":0
                                    ,"pPos":0
                                    ,"pNeg":0
                                    ,"pOutLo":0
                                    ,"pOutHi":0
                                    ,"Moda":describe["top"]
                                    ,"nModa":describe["freq"]
                                    ,"pModa":round((describe["freq"] / df.shape[0]),redondeo)}
    
    df_descripcion = pd.DataFrame([dict_descripcion])
    
    return df_descripcion
def extrae_cortes_int(df,col):
    cuts = []
    if df[col].dtype == "int64":
        cuts = df[col].drop_duplicates().to_list()
        cuts = np.unique(cuts)
        cuts = np.insert(cuts,0,-9999999)
        cuts = np.insert(cuts,0,-9999998)
        cuts = np.insert(cuts,0,-9999990)
        cuts = np.insert(cuts,0,-9000000)
        cuts = np.insert(cuts,0,np.max(cuts))
        cuts = np.sort(cuts)
        cuts = np.delete(cuts, np.where(cuts == -2))
    return cuts

def eda(df,cols,target):
    df_target = df[target]
    n = df.shape[0]
    
    df_total = pd.DataFrame(columns=["Orden","Variable","Type","Freq","Unico","Media","Desv","Min","Q1","Mediana","Q3","Max","Miss","Cero","Pos","Neg","OutLo","OutHi","pMiss","pCero","pPos","pNeg","pOutLo","pOutHi","Moda","nModa","pModa"])
    df_variables = pd.DataFrame(columns=["Orden","Variable","Orden_Cota","Cotas","Sql","Media","Total","Total_Pct"])
    
    for i in range(len(cols)):
        #print("...."+str(i+1)+". "+cols[i])
        
        # Resumen Total
        df_descripcion = descripcion_variable(df,cols[i],i+1)
        df_total = pd.concat([df_total,df_descripcion])
        
        if df[cols[i]].dtype in (["int64","float64"]):
            
            try:
                # Base inicial
                df_X = pd.DataFrame(df[cols[i]])
                df_X.loc[df_X[cols[i]].isna(),cols[i]] = -9000000
                df.loc[df[cols[i]].isna(),cols[i]] = -9000000

                # Arbol
                # model = DecisionTreeRegressor(min_samples_split = 0.20,min_samples_leaf = int(n*0.01))#,max_depth = 3
                model = DecisionTreeRegressor(min_samples_split = 0.20,min_samples_leaf = int(n*0.05))#,max_depth = 3
                model.fit(df_X,df_target)

                # Cortes del Arbol
                cuts = extrae_cortes_arbol(df,cols[i],model)
                #print(cuts)
                # Resumen de Variables
                df_bin_variable = bin_variable(cuts,df,target,cols[i],i+1)
                df_variables = pd.concat([df_variables,df_bin_variable])

            except Exception as e:
                print("...."+str(i+1)+". Error: ",e)

    df_variables = df_variables.reset_index()
    
    return df_total, df_variables

def eda_bivariado(df,cols,target,ruta_eda,name=None):
    df_target = df[target]
    n = df.shape[0]
    
    #df_total = pd.DataFrame(columns=["Orden","Variable","Type","Freq","Unico","Media","Desv","Min","Q1","Mediana","Q3","Max","Miss","Cero","Pos","Neg","OutLo","OutHi","pMiss","pCero","pPos","pNeg","pOutLo","pOutHi","Moda","nModa","pModa"])
    df_variables = pd.DataFrame(columns=["Variable","Orden_Cota","Cotas","Sql","Media","Total","Total_Pct"])
    
    #crea_carpeta(".//Exploracion_Variables")
    if name is None:  
        ruta_pdf = ruta_eda+"//02.Analisis_Bivariado.pdf"
    else:
        ruta_pdf = ruta_eda+"//02.Analisis_Bivariado"+name+".pdf"
        
    #df_descripciones = bucle_descripcion_variables(df,cols)
    
    with PdfPages(ruta_pdf) as pdf:    
        for i in range(len(cols)):
            print("...."+str(i+1)+". "+cols[i])

            # Resumen Total
            #df_total = pd.concat([df_total,df_descripcion])

            if df[cols[i]].dtype in (["int64","float64"]):

                try:
                    # Base inicial
                    df_X = pd.DataFrame(df[cols[i]])
                    df_X.loc[df_X[cols[i]].isna(),cols[i]] = -9000000
                    df.loc[df[cols[i]].isna(),cols[i]] = -9000000

                    # Arbol
                    arbol = DecisionTreeRegressor(min_samples_split = 0.20,min_samples_leaf = int(n*0.05))#,max_depth = 3
                    arbol.fit(df_X,df_target)

                    # Cortes del Arbol
                    if df[cols[i]].dtype == "int64":
                        cuts = extrae_cortes_int(df,cols[i])
                    else:
                        cuts = extrae_cortes_arbol(df,cols[i],arbol)
                    
                    # Resumen de Variables
                    df_bin_variable = bin_variable_regresion(df,cols[i],cuts,target)
                    df_variables = pd.concat([df_variables,df_bin_variable])

                    # Graficos
                    # Grafico de Barras
                    plt.figure(figsize=(15, 6), dpi=100)

                    plt.subplot(121)
                    plt.bar(df_bin_variable["Orden_Cota"],df_bin_variable["Total"],width=.5)
                    plt.xticks(df_bin_variable["Orden_Cota"])
                    plt.title(str(i+1)+". "+cols[i])
                    axes2 = plt.twinx()
                    axes2.plot(df_bin_variable["Orden_Cota"], df_bin_variable['Media'], color='Orange',marker = "o")

                    # Tabla resumen
                    plt.subplot(122)
                    cell_text = []
                    df_X = df_bin_variable[["Cotas", "Media", "Total","Total_Pct"]]
                    df_X.loc[:,"Media"] = df_X["Media"].apply(lambda x: round(x,3))
                    df_X.loc[:,"Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    #df_X["Media"] = df_X["Media"].apply(lambda x: round(x,1))
                    #df_X["Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    for row in range(len(df_X)):
                        cell_text.append(df_X.iloc[row])

                    tab = plt.table(cellText=cell_text, colLabels=df_X.columns, cellLoc='center',loc='center',colWidths=[0.6, 0.2,0.2,0.2])
                    tab.auto_set_font_size(False)
                    #tab.set_fontsize(12)
                    #tab.scale(2, 2)
                    plt.axis('off')
                    plt.tight_layout()
                    plt.subplots_adjust(left=0.1,bottom=0.1,right=0.9,top=0.9,wspace=0.25,hspace=0.4)

                    pdf.savefig()
                    plt.close()

                    # Arbol
                    fig = plt.figure(figsize=(25,20))
                    _ = tree.plot_tree(arbol, 
                                    feature_names=[cols[i]],
                                    #class_names=iris.target_names,
                                    filled=True)
                    pdf.savefig()
                    plt.close()
                except Exception as e:
                    print("...."+str(i+1)+". Error: ",e)

    df_variables = df_variables.reset_index()
    
    #return df_descripciones, df_variables    
def eda_bivariado2(df,cols,target,ruta_eda,name=None):
    df_target = df[target]
    n = df.shape[0]
    
    #df_total = pd.DataFrame(columns=["Orden","Variable","Type","Freq","Unico","Media","Desv","Min","Q1","Mediana","Q3","Max","Miss","Cero","Pos","Neg","OutLo","OutHi","pMiss","pCero","pPos","pNeg","pOutLo","pOutHi","Moda","nModa","pModa"])
    df_variables = pd.DataFrame(columns=["Variable","Orden_Cota","Cotas","Sql","Media","Total","Total_Pct"])
    
    #crea_carpeta(".//Exploracion_Variables")
    if name is None:  
        ruta_pdf = ruta_eda+"//02.Analisis_Bivariado.pdf"
    else:
        ruta_pdf = ruta_eda+"//02.Analisis_Bivariado"+name+".pdf"
        
    #df_descripciones = bucle_descripcion_variables(df,cols)
    
    with PdfPages(ruta_pdf) as pdf:    
        for i in range(len(cols)):
            print("...."+str(i+1)+". "+cols[i])

            # Resumen Total
            #df_total = pd.concat([df_total,df_descripcion])

            if df[cols[i]].dtype in (["int64","float64"]):

                try:
                    # Base inicial
                    df_X = pd.DataFrame(df[cols[i]])
                    df_X.loc[df_X[cols[i]].isna(),cols[i]] = -9000000
                    df.loc[df[cols[i]].isna(),cols[i]] = -9000000

                    # Arbol
                    arbol = DecisionTreeClassifier(min_samples_split = 0.20,min_samples_leaf = int(n*0.05))#,max_depth = 3
                    arbol.fit(df_X,df_target)

                    # Cortes del Arbol
                    if df[cols[i]].dtype == "int64":
                        cuts = extrae_cortes_int(df,cols[i])
                    else:
                        cuts = extrae_cortes_arbol(df,cols[i],arbol)
                    
                    # Resumen de Variables
                    df_bin_variable = bin_variable_regresion(df,cols[i],cuts,target)
                    df_variables = pd.concat([df_variables,df_bin_variable])

                    # Graficos
                    # Grafico de Barras
                    plt.figure(figsize=(15, 6), dpi=100)

                    plt.subplot(121)
                    plt.bar(df_bin_variable["Orden_Cota"],df_bin_variable["Total"],width=.5)
                    plt.xticks(df_bin_variable["Orden_Cota"])
                    plt.title(str(i+1)+". "+cols[i])
                    axes2 = plt.twinx()
                    axes2.plot(df_bin_variable["Orden_Cota"], df_bin_variable['Media'], color='Orange',marker = "o")

                    # Tabla resumen
                    plt.subplot(122)
                    cell_text = []
                    df_X = df_bin_variable[["Cotas", "Media", "Total","Total_Pct"]]
                    df_X.loc[:,"Media"] = df_X["Media"].apply(lambda x: round(x,3))
                    df_X.loc[:,"Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    #df_X["Media"] = df_X["Media"].apply(lambda x: round(x,1))
                    #df_X["Total_Pct"] = df_X['Total_Pct'].apply(lambda x: str(round(x*100,1))+"%")
                    for row in range(len(df_X)):
                        cell_text.append(df_X.iloc[row])

                    tab = plt.table(cellText=cell_text, colLabels=df_X.columns, cellLoc='center',loc='center',colWidths=[0.6, 0.2,0.2,0.2])
                    tab.auto_set_font_size(False)
                    #tab.set_fontsize(12)
                    #tab.scale(2, 2)
                    plt.axis('off')
                    plt.tight_layout()
                    plt.subplots_adjust(left=0.1,bottom=0.1,right=0.9,top=0.9,wspace=0.25,hspace=0.4)

                    pdf.savefig()
                    plt.close()

                    # Arbol
                    fig = plt.figure(figsize=(25,20))
                    _ = tree.plot_tree(arbol, 
                                    feature_names=[cols[i]],
                                    #class_names=iris.target_names,
                                    filled=True)
                    pdf.savefig()
                    plt.close()
                except Exception as e:
                    print("...."+str(i+1)+". Error: ",e)

    df_variables = df_variables.reset_index()

def eda_vf(df,target,var_excluidas,ruta_eda):

    start_time = time.time()
    
    df1 = df.drop(columns=var_excluidas) 
    df2 = df1

    # Correlación
    df_cor, df_cor_abs, df_cor_target = correlacion(df1,target=target)

    print("..Eda")
    #Extrae top de correlacion y columnas faltantes
    cols = df1.columns.to_list()
    cols.sort()
    cols.remove(target)
    df_cols = pd.DataFrame(cols)
    df_cols = df_cols.rename({0: "Variable"}, axis=1)
    df_cols = df_cor_target.merge(df_cols,how="outer",on="Variable")
    cols = df_cols["Variable"].to_list()

    #Descripcion Variables
    tops = df_cols["Top"].to_list()

    df_descripcion = pd.DataFrame()
    for i in range(len(cols)):
        df_descripcion = pd.concat([df_descripcion,descripcion_variable(df1,cols[i],tops[i],redondeo=None)])

    #Inicia EDA
    df1, df_variables = eda(df1,cols,target)

    with pd.ExcelWriter(ruta_eda+"\\01.Analisis_Exploratorio.xlsx") as writer:  
        df_cor.to_excel(writer, sheet_name='Correlacion')
        df_cor_abs.to_excel(writer, sheet_name='Correlacion_Abs')
        df_cor_target.to_excel(writer, sheet_name='Correlacion_Target',index=False)
        df_descripcion.to_excel(writer, sheet_name='EDA',index=False)
        df_variables.to_excel(writer, sheet_name='Variables',index=False)
        #df.to_excel(writer,sheet_name='diccionario',index=False)

    #Inicia EDA bivariado
    print("...Bivariado:")
    eda_bivariado(df2,cols,target,ruta_eda)   
    elapsed_time = time.time() - start_time
    print(f"Tiempo transcurrido: {int(elapsed_time // 60)} minutos y {elapsed_time % 60:.3f} segundos")
    start_time = time.time()

def eda_vf2(df,target,var_excluidas,var_categoricas,ruta_eda):

    start_time = time.time()
    
    df = df.drop(columns=var_excluidas) # quitando variables
    df1 = df.drop(columns=var_categoricas) 
    df2 = df1
    df_cat = df[[target]+var_categoricas]

    # Correlación
    df_cor, df_cor_abs, df_cor_target = correlacion(df1,target=target)

    print("..Eda")
    #Extrae top de correlacion y columnas faltantes
    cols = df1.columns.to_list()
    cols.sort()
    cols.remove(target)
    df_cols = pd.DataFrame(cols)
    df_cols = df_cols.rename({0: "Variable"}, axis=1)
    df_cols = df_cor_target.merge(df_cols,how="outer",on="Variable")
    cols = df_cols["Variable"].to_list()

    #Descripcion Variables
    tops = df_cols["Top"].to_list()

    df_descripcion = pd.DataFrame()
    for i in range(len(cols)):
        df_descripcion = pd.concat([df_descripcion,descripcion_variable(df1,cols[i],tops[i],redondeo=None)])

    #Inicia EDA
    df1, df_variables = eda(df1,cols,target)

    ########################################################################
    # Para var. categóricas

    var_cat = df_cat.columns.to_list()
    var_cat.sort()
    var_cat.remove(target)
    df_cols_cat = pd.DataFrame(var_cat)
    df_cols_cat = df_cols_cat.rename({0: "Variable"}, axis=1)
    var_cat = df_cols_cat["Variable"].to_list()

    #Descripcion Variables cat.
    df_descripcion_cat = pd.DataFrame()
    for i in range(len(var_cat)):
        df_descripcion_cat = pd.concat([df_descripcion_cat,descripcion_variable(df_cat,var_cat[i],var_cat[i],redondeo=None)])

    ########################################################################

    with pd.ExcelWriter(ruta_eda+"\\01.Analisis_Exploratorio.xlsx") as writer:  
        df_cor.to_excel(writer, sheet_name='Correlacion')
        df_cor_abs.to_excel(writer, sheet_name='Correlacion_Abs')
        df_cor_target.to_excel(writer, sheet_name='Correlacion_Target',index=False)
        df_descripcion.to_excel(writer, sheet_name='EDA',index=False)
        df_descripcion_cat.to_excel(writer, sheet_name='EDA_cat',index=False)
        df_variables.to_excel(writer, sheet_name='Variables',index=False)
        #df.to_excel(writer,sheet_name='diccionario',index=False)

    #Inicia EDA bivariado
    print("...Bivariado:")
    eda_bivariado(df2,cols,target,ruta_eda)   
    elapsed_time = time.time() - start_time
    print(f"Tiempo transcurrido: {int(elapsed_time // 60)} minutos y {elapsed_time % 60:.3f} segundos")
    start_time = time.time()

def eda_vf3(df,target,var_excluidas,var_categoricas,ruta_eda):

    start_time = time.time()
    
    df = df.drop(columns=var_excluidas) # quitando variables
    df1 = df.drop(columns=var_categoricas) 
    df2 = df1
    df_cat = df[[target]+var_categoricas]

    # Correlación
    df_cor, df_cor_abs, df_cor_target = correlacion(df1,target=target)

    print("..Eda")
    #Extrae top de correlacion y columnas faltantes
    cols = df1.columns.to_list()
    cols.sort()
    cols.remove(target)
    df_cols = pd.DataFrame(cols)
    df_cols = df_cols.rename({0: "Variable"}, axis=1)
    df_cols = df_cor_target.merge(df_cols,how="outer",on="Variable")
    cols = df_cols["Variable"].to_list()

    #Descripcion Variables
    tops = df_cols["Top"].to_list()

    df_descripcion = pd.DataFrame()
    for i in range(len(cols)):
        df_descripcion = pd.concat([df_descripcion,descripcion_variable(df1,cols[i],tops[i],redondeo=None)])

    #Inicia EDA
    df1, df_variables = eda(df1,cols,target)

    ########################################################################
    # Para var. categóricas

    var_cat = df_cat.columns.to_list()
    var_cat.sort()
    var_cat.remove(target)
    df_cols_cat = pd.DataFrame(var_cat)
    df_cols_cat = df_cols_cat.rename({0: "Variable"}, axis=1)
    var_cat = df_cols_cat["Variable"].to_list()

    #Descripcion Variables cat.
    df_descripcion_cat = pd.DataFrame()
    for i in range(len(var_cat)):
        df_descripcion_cat = pd.concat([df_descripcion_cat,descripcion_variable(df_cat,var_cat[i],var_cat[i],redondeo=None)])

    ########################################################################

    with pd.ExcelWriter(ruta_eda+"\\01.Analisis_Exploratorio.xlsx") as writer:  
        df_cor.to_excel(writer, sheet_name='Correlacion')
        df_cor_abs.to_excel(writer, sheet_name='Correlacion_Abs')
        df_cor_target.to_excel(writer, sheet_name='Correlacion_Target',index=False)
        df_descripcion.to_excel(writer, sheet_name='EDA',index=False)
        df_descripcion_cat.to_excel(writer, sheet_name='EDA_cat',index=False)
        df_variables.to_excel(writer, sheet_name='Variables',index=False)
        #df.to_excel(writer,sheet_name='diccionario',index=False)

    #Inicia EDA bivariado
    print("...Bivariado:")
    eda_bivariado2(df2,cols,target,ruta_eda)   
    elapsed_time = time.time() - start_time
    print(f"Tiempo transcurrido: {int(elapsed_time // 60)} minutos y {elapsed_time % 60:.3f} segundos")
    start_time = time.time()

def coincidencia(y_real_train, y_pred_train, umbral, index):
    
    porc_pres_train = ((y_pred_train - y_real_train)/y_real_train)

    # Definir las condiciones para la clasificación
    condiciones_train = [
        (porc_pres_train >= -umbral) & (porc_pres_train <= umbral),  # Coincidencia
        porc_pres_train > umbral,  # Sobreestimación
        porc_pres_train < -umbral  # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Crea el vector de clasificación
    clasificacion_train = np.select(condiciones_train, clases, default='No clasificado')

    # Crear DataFrames con los resultados
    df_precision_train = pd.DataFrame({'y_real': y_real_train, 'y_predicho': y_pred_train, 'porc_pres': porc_pres_train, 'clas_pres': clasificacion_train})
    
    # Calcular las proporciones de cada clase
    presc_train = df_precision_train['clas_pres'].value_counts(normalize=True)

    # Crear un DataFrame con los resultados finales
    resultados = {
        'Subestimacion': [presc_train.get('Subestimacion', 0)],
        'Coincidencia': [presc_train.get('Coincidencia', 0)],
        'Sobreestimacion': [presc_train.get('Sobreestimacion', 0)]
    }
    df_resultados_precs = pd.DataFrame(resultados, index=[index])
    
    return df_resultados_precs

def calcular_coincidencia(y_pred_train, y_pred_test, y_real_train, y_real_test, umbral):
    # Evitar errores dividiendo por cero
    #y_real_train = np.where(y_real_train == 0, np.nan, y_real_train)
    #y_real_test = np.where(y_real_test == 0, np.nan, y_real_test)
    
    # Calcular el porcentaje de error
    porc_pres_train = ((y_pred_train - y_real_train)/y_real_train)
    porc_pres_test = ((y_pred_test - y_real_test)/y_real_test)

    # Definir las condiciones para la clasificación
    condiciones_train = [
        (porc_pres_train >= -umbral) & (porc_pres_train <= umbral),  # Coincidencia
        porc_pres_train > umbral,  # Sobreestimación
        porc_pres_train < -umbral  # Subestimación
    ]
    condiciones_test = [
        (porc_pres_test >= -umbral) & (porc_pres_test <= umbral),  # Coincidencia
        porc_pres_test > umbral,  # Sobreestimación
        porc_pres_test < -umbral  # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Crea el vector de clasificación
    clasificacion_train = np.select(condiciones_train, clases, default='No clasificado')
    clasificacion_test = np.select(condiciones_test, clases, default='No clasificado')

    # Crear DataFrames con los resultados
    df_precision_train = pd.DataFrame({'y_real': y_real_train, 'y_predicho': y_pred_train, 'porc_pres': porc_pres_train, 'clas_pres': clasificacion_train})
    df_precision_test = pd.DataFrame({'y_real': y_real_test, 'y_predicho': y_pred_test, 'porc_pres': porc_pres_test, 'clas_pres': clasificacion_test})

    # Calcular las proporciones de cada clase
    presc_train = df_precision_train['clas_pres'].value_counts(normalize=True)
    presc_test = df_precision_test['clas_pres'].value_counts(normalize=True)

    # Crear un DataFrame con los resultados finales
    resultados = {
        'Subestimacion': [presc_train.get('Subestimacion', 0), presc_test.get('Subestimacion', 0)],
        'Coincidencia': [presc_train.get('Coincidencia', 0), presc_test.get('Coincidencia', 0)],
        'Sobreestimacion': [presc_train.get('Sobreestimacion', 0), presc_test.get('Sobreestimacion', 0)]
    }
    df_resultados_precs = pd.DataFrame(resultados, index=['Train', 'Test'])
    
    return df_resultados_precs

def calcular_coincidencia_v2(y_real_train,y_pred_train, y_real_test,y_pred_test, y_real_oot1,y_pred_oot1, y_real_oot2,y_pred_oot2, y_real_oot3,y_pred_oot3, umbral):

    # Calcular el porcentaje de error
    porc_pres_train = ((y_pred_train - y_real_train)/y_real_train)
    porc_pres_test = ((y_pred_test - y_real_test)/y_real_test)
    porc_pres_oot1 = ((y_pred_oot1 - y_real_oot1)/y_real_oot1)
    porc_pres_oot2 = ((y_pred_oot2 - y_real_oot2)/y_real_oot2)
    porc_pres_oot3 = ((y_pred_oot3 - y_real_oot3)/y_real_oot3)

    # Definir las condiciones para la clasificación
    condiciones_train = [
        (porc_pres_train >= -umbral) & (porc_pres_train <= umbral),  # Coincidencia
        porc_pres_train > umbral,  # Sobreestimación
        porc_pres_train < -umbral  # Subestimación
    ]
    condiciones_test = [
        (porc_pres_test >= -umbral) & (porc_pres_test <= umbral),  # Coincidencia
        porc_pres_test > umbral,  # Sobreestimación
        porc_pres_test < -umbral  # Subestimación
    ]
    condiciones_oot1 = [
        (porc_pres_oot1 >= -umbral) & (porc_pres_oot1 <= umbral),  # Coincidencia
        porc_pres_oot1 > umbral,  # Sobreestimación
        porc_pres_oot1 < -umbral  # Subestimación
    ]
    condiciones_oot2 = [
        (porc_pres_oot2 >= -umbral) & (porc_pres_oot2 <= umbral),  # Coincidencia
        porc_pres_oot2 > umbral,  # Sobreestimación
        porc_pres_oot2 < -umbral  # Subestimación
    ]
    condiciones_oot3 = [
        (porc_pres_oot3 >= -umbral) & (porc_pres_oot3 <= umbral),  # Coincidencia
        porc_pres_oot3 > umbral,  # Sobreestimación
        porc_pres_oot3 < -umbral  # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Crea el vector de clasificación
    clasificacion_train = np.select(condiciones_train, clases, default='No clasificado')
    clasificacion_test = np.select(condiciones_test, clases, default='No clasificado')
    clasificacion_oot1 = np.select(condiciones_oot1, clases, default='No clasificado')
    clasificacion_oot2 = np.select(condiciones_oot2, clases, default='No clasificado')
    clasificacion_oot3 = np.select(condiciones_oot3, clases, default='No clasificado')

    # Crear DataFrames con los resultados
    df_precision_train = pd.DataFrame({'y_real': y_real_train, 'y_predicho': y_pred_train, 'porc_pres': porc_pres_train, 'clas_pres': clasificacion_train})
    df_precision_test = pd.DataFrame({'y_real': y_real_test, 'y_predicho': y_pred_test, 'porc_pres': porc_pres_test, 'clas_pres': clasificacion_test})
    df_precision_oot1 = pd.DataFrame({'y_real': y_real_oot1, 'y_predicho': y_pred_oot1, 'porc_pres': porc_pres_oot1, 'clas_pres': clasificacion_oot1})
    df_precision_oot2 = pd.DataFrame({'y_real': y_real_oot2, 'y_predicho': y_pred_oot2, 'porc_pres': porc_pres_oot2, 'clas_pres': clasificacion_oot2})
    df_precision_oot3 = pd.DataFrame({'y_real': y_real_oot3, 'y_predicho': y_pred_oot3, 'porc_pres': porc_pres_oot3, 'clas_pres': clasificacion_oot3})

    # Calcular las proporciones de cada clase
    presc_train = df_precision_train['clas_pres'].value_counts(normalize=True)
    presc_test = df_precision_test['clas_pres'].value_counts(normalize=True)
    presc_oot1= df_precision_oot1['clas_pres'].value_counts(normalize=True)
    presc_oot2 = df_precision_oot2['clas_pres'].value_counts(normalize=True)
    presc_oot3 = df_precision_oot3['clas_pres'].value_counts(normalize=True)

    # Crear un DataFrame con los resultados finales
    resultados = {
        'Subestimacion': [presc_train.get('Subestimacion', 0), presc_test.get('Subestimacion', 0), presc_oot1.get('Subestimacion', 0), presc_oot2.get('Subestimacion', 0), presc_oot3.get('Subestimacion', 0)],
        'Coincidencia': [presc_train.get('Coincidencia', 0), presc_test.get('Coincidencia', 0), presc_oot1.get('Coincidencia', 0), presc_oot2.get('Coincidencia', 0), presc_oot3.get('Coincidencia', 0)],
        'Sobreestimacion': [presc_train.get('Sobreestimacion', 0), presc_test.get('Sobreestimacion', 0), presc_oot1.get('Sobreestimacion', 0), presc_oot2.get('Sobreestimacion', 0), presc_oot3.get('Sobreestimacion', 0)]
    }
    df_resultados_precs = pd.DataFrame(resultados, index=['Train', 'Test', 'OOT1', 'OOT2', 'OOT3'])
    
    return df_resultados_precs

def calcular_coincidencia_v0(y_pred_train, y_real_train, umbral):
    
    # Calcular el porcentaje de error
    porc_pres_train = ((y_pred_train - y_real_train)/y_real_train)

    # Definir las condiciones para la clasificación
    condiciones_train = [
        (porc_pres_train >= -umbral) & (porc_pres_train <= umbral),  # Coincidencia
        porc_pres_train > umbral,  # Sobreestimación
        porc_pres_train < -umbral  # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Crea el vector de clasificación
    clasificacion_train = np.select(condiciones_train, clases, default='No clasificado')

    # Crear DataFrames con los resultados
    df_precision_train = pd.DataFrame({'y_real': y_real_train, 'y_predicho': y_pred_train, 'porc_pres': porc_pres_train, 'clas_pres': clasificacion_train})

    # Calcular las proporciones de cada clase
    presc_train = df_precision_train['clas_pres'].value_counts(normalize=True)

    # Crear un DataFrame con los resultados finales
    resultados = {
        'Subestimacion': [presc_train.get('Subestimacion', 0)],
        'Coincidencia': [presc_train.get('Coincidencia', 0)],
        'Sobreestimacion': [presc_train.get('Sobreestimacion', 0)]
    }
    df_resultados_precs = pd.DataFrame(resultados, index=['Train'])
    
    return df_resultados_precs

def calcular_precision_x_intervalos(y_pred, y_real, umbral, bins):
    import numpy as np
    import pandas as pd

    porc_pres = (y_pred - y_real) / y_real
    condiciones = [
        (porc_pres < -umbral),              # Subestimación
        (porc_pres >= -umbral) & (porc_pres <= umbral),  # Coincidencia
        (porc_pres > umbral)                # Sobreestimación
    ]
    clases = ['Subestimacion', 'Coincidencia', 'Sobreestimacion']

    clasificacion = np.select(condiciones, clases, default='No clasificado')
    df_precision = pd.DataFrame({
        'y_real': y_real,
        'y_predicho': y_pred,
        'porc_pres': porc_pres,
        'clas_pres': clasificacion
    })

    df_precision['Intervalos'] = pd.cut(df_precision['y_real'], bins)

    frecuencia = df_precision['Intervalos'].value_counts().sort_index()
    porcentaje = frecuencia / frecuencia.sum()
    prom_real= df_precision.groupby('Intervalos')['y_real'].mean()
    prom_pred = df_precision.groupby('Intervalos')['y_predicho'].mean()

    resultado = pd.DataFrame({
        'Casos': frecuencia,
        'Casos (%)': porcentaje,
        'Prom. real': prom_real,
        'Prom. pred': prom_pred
    })

    distribucion = df_precision.groupby(['Intervalos', 'clas_pres']).size().unstack(fill_value=0)

    # Asegurar el orden deseado de las columnas
    for col in ['Subestimacion', 'Coincidencia', 'Sobreestimacion']:
        if col not in distribucion.columns:
            distribucion[col] = 0
    distribucion = distribucion[['Subestimacion', 'Coincidencia', 'Sobreestimacion']]

    distribucion = distribucion.div(distribucion.sum(axis=1), axis=0)

    resultado_final = resultado.join(distribucion, on='Intervalos')

    return resultado_final

def calcular_precision_x_intervalos_v2(y_pred, y_real, umbral, bins):

    def rmspe(y_true, y_pred):
        return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

    porc_pres = (y_pred - y_real) / y_real
    condiciones = [
        (porc_pres < -umbral),              # Subestimación
        (porc_pres >= -umbral) & (porc_pres <= umbral),  # Coincidencia
        (porc_pres > umbral)                # Sobreestimación
    ]
    clases = ['Subestimacion', 'Coincidencia', 'Sobreestimacion']

    clasificacion = np.select(condiciones, clases, default='No clasificado')
    df_precision = pd.DataFrame({
        'y_real': y_real,
        'y_predicho': y_pred,
        'porc_pres': porc_pres,
        'clas_pres': clasificacion
    })

    df_precision['Intervalos'] = pd.cut(df_precision['y_real'], bins)

    frecuencia = df_precision['Intervalos'].value_counts().sort_index()
    porcentaje = frecuencia / frecuencia.sum()
    prom_real= df_precision.groupby('Intervalos')['y_real'].mean()
    prom_pred = df_precision.groupby('Intervalos')['y_predicho'].mean()
    rmspe_cat = df_precision.groupby('Intervalos').apply(lambda df: rmspe(df['y_real'], df['y_predicho']) )

    resultado = pd.DataFrame({
        'Casos': frecuencia,
        'Casos (%)': porcentaje,
        'Prom. real': prom_real,
        'Prom. pred': prom_pred,
        'RMSPE': rmspe_cat
    })

    distribucion = df_precision.groupby(['Intervalos', 'clas_pres']).size().unstack(fill_value=0)

    # Asegurar el orden deseado de las columnas
    for col in ['Subestimacion', 'Coincidencia', 'Sobreestimacion']:
        if col not in distribucion.columns:
            distribucion[col] = 0
    distribucion = distribucion[['Subestimacion', 'Coincidencia', 'Sobreestimacion']]

    distribucion = distribucion.div(distribucion.sum(axis=1), axis=0)

    resultado_final = resultado.join(distribucion, on='Intervalos')

    return resultado_final

def calcular_precision_x_intervalos_v3(y_pred, y_real, umbral, bins):

    from sklearn import metrics

    def rmspe(y_true, y_pred):
        return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

    porc_pres = (y_pred - y_real) / y_real
    condiciones = [
        (porc_pres < -umbral),              # Subestimación
        (porc_pres >= -umbral) & (porc_pres <= umbral),  # Coincidencia
        (porc_pres > umbral)                # Sobreestimación
    ]
    clases = ['Subestimacion', 'Coincidencia', 'Sobreestimacion']

    clasificacion = np.select(condiciones, clases, default='No clasificado')
    df_precision = pd.DataFrame({
        'y_real': y_real,
        'y_predicho': y_pred,
        'porc_pres': porc_pres,
        'clas_pres': clasificacion
    })

    df_precision['Intervalos'] = pd.cut(df_precision['y_real'], bins)

    frecuencia = df_precision['Intervalos'].value_counts().sort_index()
    porcentaje = frecuencia / frecuencia.sum()
    prom_real= df_precision.groupby('Intervalos')['y_real'].mean()
    prom_pred = df_precision.groupby('Intervalos')['y_predicho'].mean()
    mae_cat = df_precision.groupby('Intervalos').apply(lambda df: metrics.mean_absolute_error(df['y_real'], df['y_predicho']) )
    mape_cat = df_precision.groupby('Intervalos').apply(lambda df: metrics.mean_absolute_percentage_error(df['y_real'], df['y_predicho']) )
    rmspe_cat = df_precision.groupby('Intervalos').apply(lambda df: rmspe(df['y_real'], df['y_predicho']) )

    resultado = pd.DataFrame({
        'Casos': frecuencia,
        'Casos (%)': porcentaje,
        'Prom. real': prom_real,
        'Prom. pred': prom_pred,
        'MAE': mae_cat,
        'MAPE': mape_cat,
        'RMSPE': rmspe_cat
    })

    distribucion = df_precision.groupby(['Intervalos', 'clas_pres']).size().unstack(fill_value=0)

    # Asegurar el orden deseado de las columnas
    for col in ['Subestimacion', 'Coincidencia', 'Sobreestimacion']:
        if col not in distribucion.columns:
            distribucion[col] = 0
    distribucion = distribucion[['Subestimacion', 'Coincidencia', 'Sobreestimacion']]

    distribucion = distribucion.div(distribucion.sum(axis=1), axis=0)

    resultado_final = resultado.join(distribucion, on='Intervalos')

    return resultado_final

def calcular_precision_x_categorias_segmento(y_pred, y_real, df_segmento, umbral):
    # Calcular el porcentaje de precisión
    porc_pres = (y_pred - y_real) / y_real

    # Definir condiciones para clasificar
    condiciones = [
        (porc_pres >= -umbral) & (porc_pres <= umbral),  # Coincidencia
        (porc_pres > umbral),                            # Sobreestimación
        (porc_pres < -umbral)                            # Subestimación
    ]
    clases = ['Coincidencia', 'Sobreestimacion', 'Subestimacion']

    # Clasificar según las condiciones
    clasificacion = np.select(condiciones, clases, default='No clasificado')

    # Crear DataFrame con resultados
    df_precision = pd.DataFrame({
        'y_real': y_real,
        'y_predicho': y_pred,
        'porc_pres': porc_pres,
        'clas_pres': clasificacion,
        'Categoría': df_segmento
    })

    # Calcular frecuencia y promedio por categoría
    frecuencia = df_precision['Categoría'].value_counts().sort_index()
    frecuencia_porc = frecuencia / frecuencia.sum()
    prom_real = df_precision.groupby('Categoría')['y_real'].mean().sort_index()
    prom_pred = df_precision.groupby('Categoría')['y_predicho'].mean().sort_index()

    resultado = pd.DataFrame({
        'Casos': frecuencia,
        'Casos (%)': frecuencia_porc,
        'Prom. real': prom_real,
        'Prom. pred.': prom_pred
    })

    # Distribución por clase de precisión
    distribucion = df_precision.groupby(['Categoría', 'clas_pres']).size().unstack(fill_value=0)

    # Asegurar columnas en orden deseado
    for col in ['Subestimacion', 'Coincidencia', 'Sobreestimacion']:
        if col not in distribucion.columns:
            distribucion[col] = 0
    distribucion = distribucion[['Subestimacion', 'Coincidencia', 'Sobreestimacion']]

    # Calcular porcentajes
    distribucion = distribucion.div(distribucion.sum(axis=1), axis=0)

    # Unir resultados
    resultado_final = resultado.join(distribucion, on='Categoría')

    return resultado_final

def Arbol_Regresion(df,X_names,y_name,porc_min_samples_leaf,omitir_nulos=True,mostrar_prop=True):
    """
    Procesa un Arbol de Regresion.
    
    :param df: DataFrame a procesar.
    :param X_names: Nombre de la/s variable/s independientes.
    :param y_name: Nombre de la variable objetivo.
    :param porc_min_samples_leaf: Porcentaje mínimo del total de las muestras en las hojas.
    :param omitir_nulos: Booleano que indica si se deben omitir los valores nulos.
    :param mostrar_prop: Booleano que indica si se quiere mostrar las muestras como proporcion.
    :return: Grafico del arbol de regresion.
    """
    df_ar = df

    if omitir_nulos:
        df_ar = df_ar.dropna(subset=X_names)
    else:
        df_ar

    X = df_ar[X_names]
    y = df_ar[y_name]

    regressor = DecisionTreeRegressor(random_state = 0,max_depth=5, min_samples_leaf = int(len(X) * porc_min_samples_leaf))
    regressor.fit(X, y)
    # Gráfico
    fig, ax = plt.subplots(figsize=(20,10))

    print(f"Profundidad del árbol: {regressor.get_depth()}")
    print(f"Número de nodos terminales: {regressor.get_n_leaves()}")

    plot = plot_tree(
                decision_tree = regressor,
                feature_names = X.columns,
                class_names   = [y.name],
                proportion    = mostrar_prop,
                filled        = True,
                impurity      = False,
                fontsize      = 10,
                precision     = 8,
                ax            = ax
            )   

def Arbol_Clasificacion_v3(df, X_names, y_name, porc_min_samples_leaf=0.05,max_depth=5,class_weight=None, omitir_nulos=True, mostrar_prop=True, ruta_guardado=None):
    """
    Procesa un Árbol de Clasificación.

    :param df: DataFrame a procesar.
    :param X_names: Nombre de la/s variable/s independientes.
    :param y_name: Nombre de la variable objetivo.
    :param porc_min_samples_leaf: Porcentaje mínimo del total de las muestras en las hojas.
    :param omitir_nulos: Booleano que indica si se deben omitir los valores nulos.
    :param mostrar_prop: Booleano que indica si se quiere mostrar las muestras como proporción.
    :param ruta_guardado: Ruta donde se guardará el gráfico del árbol de clasificación.
    :return: Gráfico del árbol de clasificación y DataFrame con los puntos de corte.
    """
    df_ar = df.copy()
    df_ar[y_name] = pd.Categorical(df_ar[y_name])

    if omitir_nulos:
        df_ar = df_ar.dropna(subset=X_names)
    else:
        df_ar

    X = df_ar[X_names]
    y = df_ar[y_name]

    classifier = DecisionTreeClassifier(random_state=0, max_depth=max_depth, min_samples_leaf=int(len(X) * porc_min_samples_leaf),class_weight=class_weight)
    classifier.fit(X, y)
    
    # Gráfico
    fig, ax = plt.subplots(figsize=(20, 10))

    print(f"Clases en y_train: {classifier.classes_}")
    class_names = [str(cls) for cls in classifier.classes_]
    print(f"Profundidad del árbol: {classifier.get_depth()}")
    print(f"Número de nodos terminales: {classifier.get_n_leaves()}")

    plot_tree(
        decision_tree=classifier,
        feature_names=X.columns,
        class_names=class_names,
        proportion=mostrar_prop,
        filled=True,
        impurity=False,
        fontsize=10,
        precision=4,
        ax=ax
    )

    if ruta_guardado:
        ruta_completa = ruta_guardado + 'arbol_clasif_' + "_".join(X_names) + '.png'
        fig.savefig(ruta_completa, format='png')

    # Crear DataFrame con los puntos de corte
    n_nodes = classifier.tree_.node_count
    children_left = classifier.tree_.children_left
    children_right = classifier.tree_.children_right
    feature = classifier.tree_.feature
    threshold = classifier.tree_.threshold

    node_depth = np.zeros(shape=n_nodes, dtype=np.int64)
    is_leaves = np.zeros(shape=n_nodes, dtype=bool)
    stack = [(0, 0)]  # seed is the root node id and its depth
    while len(stack) > 0:
        node_id, depth = stack.pop()
        node_depth[node_id] = depth

        # If we have a test node
        if (children_left[node_id] != children_right[node_id]):
            stack.append((children_left[node_id], depth + 1))
            stack.append((children_right[node_id], depth + 1))
        else:
            is_leaves[node_id] = True

    cut_points = []
    for i in range(n_nodes):
        if not is_leaves[i]:
            cut_points.append({
                'node': i,
                'feature': X.columns[feature[i]],
                'threshold': threshold[i],
                'depth': node_depth[i]
            })

    df_cut_points = pd.DataFrame(cut_points)
    return fig, df_cut_points

def box_plot_outliers(data,list_var_num):
    # Crear subplots para mostrar gráficos al lado de otro con un límite de hasta seis boxplots por fila
    df = data[list_var_num]
    variables = df.columns
    num_vars = len(variables)
    num_cols = min(6, num_vars)
    num_rows = (num_vars + num_cols - 1) // num_cols

    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(18, num_rows * 3.5))

    # Asegurarse de que axes sea un array bidimensional
    axes = np.array(axes).reshape(num_rows, num_cols)

    for i, variable in enumerate(variables):
        row = i // num_cols
        col = i % num_cols
        df.boxplot(column=variable, ax=axes[row, col])
        axes[row, col].set_title(f'{variable}')
        axes[row, col].set_xlabel('Variables')
        axes[row, col].set_ylabel('Valores')

    # Eliminar subplots vacíos si hay menos variables que subplots
    for j in range(i + 1, num_rows * num_cols):
        fig.delaxes(axes.flatten()[j])

    plt.tight_layout()
    plt.show()

#   def metricas_clasificacion(y_train_real, y_train_predicho, y_test_real, y_test_predicho, plot_confusion_matrix=False):
#       # Train
#       accuracy_train = metrics.accuracy_score(y_train_real, y_train_predicho)
#       auc_train = roc_auc_score(y_train_real, y_train_predicho)
#       precision_train = metrics.precision_score(y_train_real, y_train_predicho, average='weighted')
#       recall_train = metrics.recall_score(y_train_real, y_train_predicho, average='weighted')
#       f1_train = metrics.f1_score(y_train_real, y_train_predicho, average='weighted')
#       kappa_train = cohen_kappa_score(y_train_real, y_train_predicho)
#       gini_train = 2 * auc_train - 1
#       
#       # Test
#       accuracy_test = metrics.accuracy_score(y_test_real, y_test_predicho)
#       auc_test = roc_auc_score(y_test_real, y_test_predicho)
#       precision_test = metrics.precision_score(y_test_real, y_test_predicho, average='weighted')
#       recall_test = metrics.recall_score(y_test_real, y_test_predicho, average='weighted')
#       f1_test = metrics.f1_score(y_test_real, y_test_predicho, average='weighted')
#       kappa_test = cohen_kappa_score(y_test_real, y_test_predicho)
#       gini_test = 2 * auc_test - 1
#       
#       # Resumen
#       resultados = {
#           'Accuracy': [accuracy_train, accuracy_test],
#           'AUC': [auc_train, auc_test],
#           'Recall': [recall_train, recall_test],
#           'Precision': [precision_train, precision_test],
#           'F1 Score': [f1_train, f1_test],
#           'Kappa': [kappa_train, kappa_test],
#           'Gini': [gini_train, gini_test]
#       }
#       resultados = pd.DataFrame(resultados, index=['Train', 'Test'])
#       
#       if plot_confusion_matrix:
#           fig, axs = plt.subplots(1, 2, figsize=(10, 5))
#           
#           # Primer gráfico: Matriz de confusión para el conjunto de entrenamiento
#           metrics.plot_confusion_matrix(y_train_real, y_train_predicho, ax=axs[0], cmap='Blues')
#           axs[0].set_title('Train Confusion Matrix')
#           
#           # Segundo gráfico: Matriz de confusión para el conjunto de prueba
#           metrics.plot_confusion_matrix(y_test_real, y_test_predicho, ax=axs[1], cmap='Blues')
#           axs[1].set_title('Test Confusion Matrix')
#           
#           plt.tight_layout()
#       
#       return resultados


# Se corrige el calculo de auc, gini
def metricas_clasificacion(y_train_real, y_train_pred, y_test_real, y_test_pred, p_corte,average = 'binary', plot_confusion_matrix=False):

    y_train_predicho = (y_train_pred >= p_corte).astype(int)
    y_test_predicho = (y_test_pred >= p_corte).astype(int)

    # Train
    accuracy_train = metrics.accuracy_score(y_train_real, y_train_predicho)
    auc_train = roc_auc_score(y_train_real, y_train_pred)
    precision_train = metrics.precision_score(y_train_real, y_train_predicho, average=average)
    recall_train = metrics.recall_score(y_train_real, y_train_predicho, average=average)
    f1_train = metrics.f1_score(y_train_real, y_train_predicho, average=average)
    kappa_train = cohen_kappa_score(y_train_real, y_train_predicho)
    gini_train = 2 * auc_train - 1
    
    # Test
    accuracy_test = metrics.accuracy_score(y_test_real, y_test_predicho)
    auc_test = roc_auc_score(y_test_real, y_test_pred)
    precision_test = metrics.precision_score(y_test_real, y_test_predicho, average=average)
    recall_test = metrics.recall_score(y_test_real, y_test_predicho, average=average)
    f1_test = metrics.f1_score(y_test_real, y_test_predicho, average=average)
    kappa_test = cohen_kappa_score(y_test_real, y_test_predicho)
    gini_test = 2 * auc_test - 1
    
    # Resumen
    resultados = {
        'Accuracy': [accuracy_train, accuracy_test],
        'AUC': [auc_train, auc_test],
        'Recall': [recall_train, recall_test],
        'Precision': [precision_train, precision_test],
        'F1 Score': [f1_train, f1_test],
        'Kappa': [kappa_train, kappa_test],
        'Gini': [gini_train, gini_test]
    }
    resultados = pd.DataFrame(resultados, index=['Train', 'Test'])
    
    if plot_confusion_matrix:
        fig, axs = plt.subplots(1, 2, figsize=(10, 5))
        
        # Primer gráfico: Matriz de confusión para el conjunto de entrenamiento
        metrics.plot_confusion_matrix(y_train_real, y_train_predicho, ax=axs[0], cmap='Blues')
        axs[0].set_title('Train Confusion Matrix')
        
        # Segundo gráfico: Matriz de confusión para el conjunto de prueba
        metrics.plot_confusion_matrix(y_test_real, y_test_predicho, ax=axs[1], cmap='Blues')
        axs[1].set_title('Test Confusion Matrix')
        
        plt.tight_layout()
    
    return resultados

#   def calcular_casos_x_deciles(df, variable, puntos_de_corte=None):
#       tiene_nulos = df[variable].isnull().any()
#       
#       # Verificar si la variable es categórica
#       if df[variable].dtype == 'object' or df[variable].dtype.name == 'category':
#           puntos_de_corte = df[variable].unique().tolist()
#           cantidad_por_intervalo = df[variable].value_counts().reset_index()
#           cantidad_por_intervalo.columns = [variable, 'Cantidad']
#       else:
#           # Calcular los puntos de corte para los deciles si no se proporcionan
#           if puntos_de_corte is None:
#               puntos_de_corte = df[variable].quantile([0.1 * i for i in range(11)]).tolist()
#               puntos_de_corte = list(set(puntos_de_corte))
#               puntos_de_corte.sort()
#           
#           # Calcular la cantidad de casos por cada intervalo
#           intervalos = pd.cut(df[variable], bins=puntos_de_corte, right=False)
#           cantidad_por_intervalo = df.groupby(intervalos).size().reset_index(name='Cantidad')
#       
#       df_dec = pd.DataFrame({
#           'Intervalo': cantidad_por_intervalo[variable].astype(str),
#           'Casos': cantidad_por_intervalo['Cantidad'],
#           'Casos(%)': cantidad_por_intervalo['Cantidad'] / len(df),
#       })
#       
#       if tiene_nulos:
#           cantidad_nulos = df[variable].isnull().sum()
#           df_nulos = pd.DataFrame({'Intervalo': ['Nulos'], 'Casos': cantidad_nulos, 'Casos(%)': cantidad_nulos / len(df)})
#           df_dec = pd.concat([df_nulos, df_dec], ignore_index=True)
#       
#       return df_dec
#   
#   def calcular_psi_x_intervalo(df1, df2, variable, puntos_de_corte=None):
#       # Verificar si la variable es categórica
#       if df1[variable].dtype == 'object' or df1[variable].dtype.name == 'category':
#           puntos_de_corte = df1[variable].unique().tolist()
#       else:
#           # Calcular los puntos de corte para los deciles si no se proporcionan
#           if puntos_de_corte is None:
#               puntos_de_corte = df1[variable].quantile([0.1 * i for i in range(11)]).tolist()
#               puntos_de_corte = list(set(puntos_de_corte))  # Para eliminar posibles duplicados
#               puntos_de_corte.sort()
#   
#       prop_1 = calcular_casos_x_deciles(df1, variable, puntos_de_corte)['Casos(%)']
#       prop_2 = calcular_casos_x_deciles(df2, variable, puntos_de_corte)['Casos(%)']
#   
#       df_psi = pd.DataFrame({
#           'Intervalo': calcular_casos_x_deciles(df1, variable, puntos_de_corte)['Intervalo'].astype(str),
#           'PSI': (prop_1 - prop_2) * np.log(prop_1 / prop_2)
#       })
#   
#       suma_psi = df_psi['PSI'].sum()
#   
#       total_row = pd.DataFrame({'Intervalo': ['Total'], 'PSI': [suma_psi]})
#       df_psi = pd.concat([df_psi, total_row], ignore_index=True)
#   
#       return df_psi

def calcular_casos_x_deciles(df, variable, puntos_de_corte=None):
    #tiene_nulos = df[variable].isnull().any()
    
    # Verificar si la variable es categórica
    if df[variable].dtype == 'object' or df[variable].dtype.name == 'category':
        puntos_de_corte = df[variable].unique().tolist()
        cantidad_por_intervalo = df[variable].value_counts().reset_index()
        cantidad_por_intervalo.columns = [variable, 'Cantidad']
    else:
        # Calcular los puntos de corte para los deciles si no se proporcionan
        if puntos_de_corte is None:
            puntos_de_corte = df[variable].quantile([0.1 * i for i in range(11)]).tolist()
            puntos_de_corte = list(set(puntos_de_corte))
            puntos_de_corte.sort()
        
        # Calcular la cantidad de casos por cada intervalo
        intervalos = pd.cut(df[variable], bins=puntos_de_corte, right=False)
        cantidad_por_intervalo = df.groupby(intervalos).size().reset_index(name='Cantidad')
    
    df_dec = pd.DataFrame({
        'Intervalo': cantidad_por_intervalo[variable].astype(str),
        'Casos': cantidad_por_intervalo['Cantidad'],
        'Casos(%)': cantidad_por_intervalo['Cantidad'] / len(df),
    })
    
    #if tiene_nulos:
    #    cantidad_nulos = df[variable].isnull().sum()
    #    df_nulos = pd.DataFrame({'Intervalo': ['Nulos'], 'Casos': cantidad_nulos, 'Casos(%)': cantidad_nulos / len(df)})
    #    df_dec = pd.concat([df_nulos, df_dec], ignore_index=True)
    
    cantidad_nulos = df[variable].isnull().sum()
    df_nulos = pd.DataFrame({
        'Intervalo': ['Nulos'],
        'Casos': [cantidad_nulos],
        'Casos(%)': [cantidad_nulos / len(df)]
    })
    df_dec = pd.concat([df_nulos, df_dec], ignore_index=True)

    return df_dec

def calcular_psi_x_intervalo(df1, df2, variable, puntos_de_corte=None, epsilon=1e-10):
    # Verificar si la variable es categórica
    if df1[variable].dtype == 'object' or df1[variable].dtype.name == 'category':
        puntos_de_corte = df1[variable].unique().tolist()
    else:
        # Calcular los puntos de corte para los deciles si no se proporcionan
        if puntos_de_corte is None:
            puntos_de_corte = df1[variable].quantile([0.1 * i for i in range(11)]).tolist()
            puntos_de_corte = list(set(puntos_de_corte))  # Para eliminar posibles duplicados
            puntos_de_corte.sort()

    prop_1 = calcular_casos_x_deciles(df1, variable, puntos_de_corte)['Casos(%)']
    prop_2 = calcular_casos_x_deciles(df2, variable, puntos_de_corte)['Casos(%)']

    #  Evitar ceros para que no salga "inf"
    prop_1 = prop_1.clip(lower=epsilon)
    prop_2 = prop_2.clip(lower=epsilon)

    df_psi = pd.DataFrame({
        'Intervalo': calcular_casos_x_deciles(df1, variable, puntos_de_corte)['Intervalo'].astype(str),
        'PSI': (prop_1 - prop_2) * np.log(prop_1 / prop_2)
    })

    suma_psi = df_psi['PSI'].sum()

    total_row = pd.DataFrame({'Intervalo': ['Total'], 'PSI': [suma_psi]})
    df_psi = pd.concat([df_psi, total_row], ignore_index=True)

    return df_psi

def obtener_psi_variables(df1, df2, lista_var):
    lista_psi = []
    for variable in lista_var:
        psi = calcular_psi_x_intervalo(df1, df2, variable)['PSI'].iloc[-1]
        lista_psi.append(psi)

    return pd.DataFrame({'Variable':df1[lista_var].columns,'PSI':lista_psi})

def graficar_psi_variables(df1, df2, lista_var):
    lista_psi = []
    for variable in lista_var:
        psi = calcular_psi_x_intervalo(df1, df2, variable)['PSI'].iloc[-1]
        lista_psi.append(psi)

    psi_df = pd.DataFrame({'Variable':df1[lista_var].columns,'PSI':lista_psi})
    #psi_df['PSI'] = pd.to_numeric(psi_df['PSI'], errors='coerce').fillna(0)

    # Grafico 
    max_psi = psi_df['PSI'].max()
    if np.isfinite(max_psi):
        x_limit = max(0.3, max_psi * 1.1)
    else:
        x_limit = 0.3

    plt.figure(figsize=(10, 6))
    bars = plt.barh(psi_df['Variable'], psi_df['PSI'], color='skyblue')
    plt.axvline(x=0.1, color='orange', linestyle='--', label='Umbral Moderado')
    plt.axvline(x=0.25, color='red', linestyle='--', label='Umbral Alto')
    plt.title('Population Stability Index (PSI) por Variable')
    plt.xlabel('PSI')
    plt.ylabel('Variables')
    plt.legend(loc='upper right')
    plt.xlim(0, x_limit)

    # Agregar etiquetas a las barras
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height() / 2, f'{width:.4f}', ha='center', va='center')

    plt.tight_layout()
    plt.show()
        
    #return lista_psi

def eliminacion_recursiva_de_variables(X, y, model, list_num_features, umbral_coinc):
    
    rfe_0 = RFE(estimator=model, n_features_to_select = max(list_num_features))
    rfe_0.fit(X, y)

    variables_seleccionadas_0 = list(X.columns[rfe_0.support_])

    X = X[variables_seleccionadas_0]

    list_num_features_ord = sorted(list_num_features, reverse=True)
    list_num_features_ord.remove(max(list_num_features))
    lista_var_elim = []

    df_total = []

    for i in tqdm_notebook(list_num_features_ord):

        rfe = RFE(estimator=model, n_features_to_select=i)

        rfe.fit(X, y)

        X_train_selected = rfe.transform(X)

        model.fit(X_train_selected, y)

        y_pred_train = np.exp(model.predict(X_train_selected))

        df_metricas_rfe = metricas_regresion(np.exp(y), y_pred_train,np.exp(y), y_pred_train).loc[['Train']]
        df_coincidencias_rfe = calcular_coincidencia(y_pred_train, y_pred_train, np.exp(y), np.exp(y), umbral_coinc ).loc[['Train']]

        variables_eliminadas = list(X.columns[~rfe.support_])
        ultima_variable_eliminada = list(set(variables_eliminadas) - set(lista_var_elim))
        lista_var_elim.extend(ultima_variable_eliminada)

        df_num_var = pd.DataFrame({"Num_Variables":i,"Var_Eliminada":ultima_variable_eliminada}, index = ['Train'])
        df_concat_metricas = pd.concat([df_num_var,df_metricas_rfe, df_coincidencias_rfe], axis=1)

        df_total.append(df_concat_metricas)
    
    # Concatenando todos los dataframes de la lista
    df_total_df = pd.concat(df_total, axis = 0, ignore_index=True)
    # Invirtiendo el orden de las filas
    df_total_df = df_total_df.iloc[::-1].reset_index(drop=True)

    print(f'Número de variables seleccionadas: {rfe.n_features_}')
    variables_seleccionadas = list(X.columns[rfe.support_])
    print('\nVariables seleccionadas:', variables_seleccionadas)
    display(df_total_df)

def eliminacion_recursiva_de_variables_clas(X, y, model, list_num_features, cutoff):
    
    rfe_0 = RFE(estimator=model, n_features_to_select = max(list_num_features))
    rfe_0.fit(X, y)

    variables_seleccionadas_0 = list(X.columns[rfe_0.support_])

    X = X[variables_seleccionadas_0]

    list_num_features_ord = sorted(list_num_features, reverse=True)
    list_num_features_ord.remove(max(list_num_features))
    lista_var_elim = []

    df_total = []

    for i in tqdm_notebook(list_num_features_ord):

        rfe = RFE(estimator=clone(model), n_features_to_select=i)
        rfe.fit(X, y)

        X_train_selected = X.loc[:, rfe.support_]

        est = clone(model) # modelo LIMPIO para este fit
        est.fit(X_train_selected, y)

        y_pred_train = est.predict(X_train_selected)

        df_metricas_rfe = metricas_clasificacion(y, y_pred_train,y, y_pred_train,p_corte=cutoff).loc[['Train']]
        #df_coincidencias_rfe = calcular_coincidencia(y_pred_train, y_pred_train, np.exp(y), np.exp(y), umbral_coinc ).loc[['Train']]

        variables_eliminadas = list(X.columns[~rfe.support_])
        ultima_variable_eliminada = list(set(variables_eliminadas) - set(lista_var_elim))
        lista_var_elim.extend(ultima_variable_eliminada)

        df_num_var = pd.DataFrame({"Num_Variables":i,"Var_Eliminada":ultima_variable_eliminada}, index = ['Train'])
        #df_concat_metricas = pd.concat([df_num_var,df_metricas_rfe, df_coincidencias_rfe], axis=1)
        df_concat_metricas = pd.concat([df_num_var,df_metricas_rfe], axis=1)

        df_total.append(df_concat_metricas)
    
    # Concatenando todos los dataframes de la lista
    df_total_df = pd.concat(df_total, axis = 0, ignore_index=True)
    # Invirtiendo el orden de las filas
    df_total_df = df_total_df.iloc[::-1].reset_index(drop=True)

    print(f'Número de variables seleccionadas: {rfe.n_features_}')
    variables_seleccionadas = list(X.columns[rfe.support_])
    print('\nVariables seleccionadas:', variables_seleccionadas)
    display(df_total_df)




### sbp_modelo2